# Part A - Approach 1: Time to Purchase
**Objective and Business Relevance** 

The goal is to understand how long it takes a user to go from first viewing an item to purchasing it, whether we can predict purchase within a single session, and whether browsing intent formed in one session carries over into purchases in later sessions. 

Insights from this analysis can directly inform Versace's retargeting timing, on-site nudges, and cross-sell design.

**Structure:**
1. Exploratory Data Analysis
2. Feature Engineering
3. Survival Analysis
4. Binary Session Classification 
5. Delayed Co-purchasing Dynamics 
6. Findings & Recommendations

In [ ]:
# import sys
# !{sys.executable} -m pip install openai
# ! {sys.executable} -m pip install lightgbm
# ! {sys.executable} -m pip install scikit-survival
# ! {sys.executable} -m pip install xgboost
# ! {sys.executable} -m pip install lifelines

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
import warnings
warnings.filterwarnings('ignore')
import os

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

In [ ]:
from scipy.stats import chi2, gaussian_kde
from itertools import combinations, zip_longest
from IPython.display import display
from urllib.parse import urlparse, parse_qs
import pickle
import re

In [ ]:
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from lifelines.utils import concordance_index

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay, make_scorer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
import xgboost as xgb
import lightgbm as lgb

try:
    from sksurv.ensemble import RandomSurvivalForest
    from sksurv.util import Surv
    HAS_SKSURV = True
except ImportError:
    print('scikit-survival not installed -- run: pip install scikit-survival')
    HAS_SKSURV = False

In [ ]:
import shap

In [ ]:
ttp = pd.read_csv('IMA2026_Versace_Time_to_Purchase.csv', low_memory=False)
ttp['event_dt'] = pd.to_datetime(ttp['event_timestamp'] / 1e6, unit='s', utc=True)

print(f'TTP shape: {ttp.shape}')
print(f'Date range: {ttp["event_date"].min()} to {ttp["event_date"].max()}')
display(ttp.head(3))

# 1. Exploratory Data Analysis

In [ ]:
ttp.shape

**Checking the nature of columns**

In [ ]:
ttp.info()

In [ ]:
ttp.nunique()

In [ ]:
CARDINALITY_THRESHOLD = 20
SKIP_COLS = ['weekday', 'country_long']
for col in ttp.columns:
    if col in SKIP_COLS:
        continue
    n_unique = ttp[col].nunique(dropna=False)
    if n_unique > CARDINALITY_THRESHOLD:
        continue
    vals = ttp[col].value_counts(dropna=False).head(20)
    print(f"\n {col} ({n_unique} unique): {vals.to_string()}")

In [ ]:
cols_to_check = ['transaction_id', 'user_id']

def is_number(x):
    try:
        float(x)
        return True
    except:
        return False

for col in cols_to_check:
    vals = ttp[col].dropna().unique()
    non_numeric = [v for v in vals if not is_number(v)]
    
    print(f"\nColumn: {col}")
    print(f"Non-numeric unique values ({len(non_numeric)}):")
    print(non_numeric[:20])  # preview

In [ ]:
view_rows = ttp[ttp['event_name'] == 'view_item'].copy()
match = (view_rows['cart_value_estimate'] == view_rows['item_price']).all()
print(f'cart_value_estimate == item_price on ALL view_item rows: {match}')
print(view_rows[['item_price', 'cart_value_estimate']].describe().round(2))


On `view_item` rows, the two columns `cart_value_estimate` and `item_price` are identical across all 10,595 rows (verified below).

This means the column does **not** tell us what is already in the user's cart at the time of the event.
It is simply a re-labelled copy of the item price attached to product events.

We therefore exclude `cart_value_estimate` as a standalone feature in modelling and rely on `item_price` / `log_price` instead.

**Data summary**

In [ ]:
ttp['event_date']     = pd.to_datetime(ttp['event_date'])
ttp['event_datetime'] = pd.to_datetime(ttp['event_timestamp'] / 1e6, unit='s', utc=True)

In [ ]:
print(f"Rows           : {len(ttp):,}")
print(f"Unique users   : {ttp['user_pseudo_id'].nunique():,}")
print(f"Unique sessions: {ttp['session_id'].nunique():,}")
print(f"Event types    : {ttp['event_name'].nunique()}")
print(f"Event names    : {ttp['event_name'].unique()}")
print(f"Date range     : {ttp['event_date'].min().date()} to {ttp['event_date'].max().date()}")
print(f"Country        : {ttp['country_long'].unique().tolist()}")
# confirm all users made purchase
print(f"Purchase events: {ttp[ttp['event_name'] == 'purchase']['user_pseudo_id'].nunique():,}")
print(f'Number of purchases: {ttp[ttp["event_name"] == "purchase"].shape[0]:,}  ')

Dataset:
- covers **2 months** of Italian traffic (Jun–Jul 2025), 
- with 298k events across 911 users and 5,230 sessions
- contains **only users who completed at least one purchase**, non-converting visitors are excluded.

The 22-event taxonomy spans the full funnel from `first_visit` to `purchase`. 

With 1,364 purchases across 911 buying sessions, some sessions contain **multiple transactions**.

In [ ]:
# number of sessions that had a purchase event
print(f'Sessions with purchase: {ttp[ttp["event_name"] == "purchase"]["session_id"].nunique():,}')

## 1.1. Event types

In [ ]:
# -- event groups (controls color coding) -----------------------------------
EGROUP_COLORS = {
    "Session/Entry":   "mediumpurple",
    "Discovery":       "sandybrown",
    "Cart/Wishlist":   "mediumseagreen",
    "Checkout":        "indianred",
    "Purchase":        'steelblue',
    "Account/Utility": "gray",
}

EVENT_GROUPS = {
    "session_start":      "Session/Entry",
    "first_visit":        "Session/Entry",
    "view_item_list":     "Discovery",
    "select_item":        "Discovery",
    "view_item":          "Discovery",
    "search":             "Discovery",
    "filter":             "Discovery",
    "search_location":      "Discovery",
    "add_to_wishlist":    "Cart/Wishlist",
    "add_to_cart":        "Cart/Wishlist",
    "remove_from_cart":   "Cart/Wishlist",
    "view_cart":          "Cart/Wishlist",
    "begin_checkout":     "Checkout",
    "add_shipping_info":  "Checkout",
    "add_payment_info":   "Checkout",
    "pay_now":            "Checkout",
    "fast_checkout":      "Checkout",
    "purchase":           "Purchase",
    "login":              "Account/Utility",
    "sign_up":            "Account/Utility",
    "contact us":           "Account/Utility",
    "newsletter_complete":  "Account/Utility"
    }
GROUP_ORDER = list(EGROUP_COLORS.keys())

# -- ordered event list ------------------------------------------------------
funnel_order = list(EVENT_GROUPS.keys())
evt_counts   = ttp["event_name"].value_counts()
ordered      = [e for e in funnel_order if e in evt_counts.index]
other        = [e for e in evt_counts.index if e not in funnel_order]
evt_plot     = evt_counts[ordered + other]
bar_colors   = [EGROUP_COLORS.get(EVENT_GROUPS.get(e, "Other"), "#999999") for e in evt_plot.index]

# -- single figure, linear scale, all events including view_item_list -------
bar_colors_all = [EGROUP_COLORS.get(EVENT_GROUPS.get(e, "Other"), "#999999") for e in evt_plot.index]

fig, ax = plt.subplots(figsize=(16, 8))
ax.barh(evt_plot.index[::-1], evt_plot.values[::-1],
        color=bar_colors_all[::-1], edgecolor="white", linewidth=0.5)

ax.set_xlabel("Event count")
ax.set_title("TTP  —  Event Type Distribution", fontsize=14)

for i, v in enumerate(evt_plot.values[::-1]):
    ax.text(v + evt_plot.max() * 0.01, i, f"{v:,}", va="center", fontsize=8)

# -- color each y-tick label by its group color ------------------------------
for tick, event in zip(ax.get_yticklabels(), evt_plot.index[::-1]):
    group = EVENT_GROUPS.get(event, "Other")
    color = EGROUP_COLORS.get(group, "#999999")
    tick.set_color(color)

# -- legend ------------------------------------------------------------------
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=EGROUP_COLORS[g], label=g, edgecolor="grey", linewidth=0.5)
                  for g in GROUP_ORDER]
ax.legend(handles=legend_handles, title="Event Group",
          loc="lower right", frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
# transaction_id uses the string '(not set)' instead of NaN for non-purchase events.
# Normalise so NaN means 'not available' consistently.
ttp_norm = ttp.copy()
ttp_norm['transaction_id'] = ttp_norm['transaction_id'].replace('(not set)', np.nan)

COLS_SHOW = [
    # Group A (always present)
    'login_status', 'device_category', 'page_location',
    # Group D (sparse)
    'engagement_time_msec', 'device_model',
    # Group B (product-interaction)
    'item_id', 'item_category', 'item_price', 'item_quantity',
    'cart_value_estimate','item_revenue',
    # Group C (purchase only)
    'transaction_id',
]

event_order = funnel_order

fill_matrix = (
    ttp_norm.groupby('event_name')[COLS_SHOW]
    .apply(lambda g: g.notna().mean())
    .reindex(event_order)          # now safe — no phantom NaN rows
)

row_labels = event_order

# ── Column group colors: dark/deep palette ──────────────────────────────────
# Deliberately distinct from the brighter EGROUP_COLORS used on the y-axis.
#   Event groups: medium/vivid hues (steelblue, mediumseagreen, sandybrown…)
#   Column groups: dark/deep hues — never the same visual family.
COL_GROUP_COLORS = {
    'login_status':         'midnightblue',   # Group A — deep navy
    'device_category':      'midnightblue',
    'page_location':        'midnightblue',
    'engagement_time_msec': 'saddlebrown',   # Group D — burnt orange
    'device_model':         'saddlebrown',
    'item_id':              'darkgreen',   # Group B — dark olive green
    'item_category':        'darkgreen',
    'item_price':           'darkgreen',
    'item_quantity':        'darkgreen',
    'cart_value_estimate':  'darkgreen',
    'item_revenue':         'indigo',   # Group C — deep purple
    'transaction_id':       'indigo',
}

# ── Draw heatmap ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
sns.heatmap(
    fill_matrix,
    annot=True, fmt='.0%',
    cmap = 'PuBuGn',
    vmin=0, vmax=1,
    linewidths=0.5, linecolor='#e8e8e8',
    ax=ax,
    cbar_kws={'label': '% rows non-null', 'shrink': 0.55},
    yticklabels=row_labels)

ax.set_xticklabels(
    [col.replace('_', '\n') for col in COLS_SHOW],
    rotation=0, fontsize=8)

# Force matplotlib to instantiate all tick Text objects before coloring.
# Without this, get_yticklabels() on a seaborn heatmap returns stale/empty
# Text objects whose set_color() calls get discarded on the actual draw.
fig.canvas.draw()

# ── Color x-axis labels by column group ──────────────────────────────────────
for tick, col in zip(ax.get_xticklabels(), COLS_SHOW):
    tick.set_color(COL_GROUP_COLORS.get(col, '#333333'))
    tick.set_fontweight('bold')

# ── Color y-axis labels by event group ───────────────────────────────────────
# get_yticklabels() returns bottom→top; heatmap rows run top→bottom,
# so pair with the reversed event list.
for tick, event in zip(ax.get_yticklabels(), event_order):
    group = EVENT_GROUPS.get(event, 'Other')
    tick.set_color(EGROUP_COLORS.get(group, '#555555'))
    tick.set_fontweight('bold')

ax.tick_params(axis='x', rotation=0, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)
ax.set_title('TTP  —  Column Fill Rate by Event Type', fontsize=14, pad=14)
ax.set_xlabel('')
ax.set_ylabel('')

# ── Vertical separators between column groups ─────────────────────────────────
ax.axvline(3,  color='#444', linewidth=2.0)   # A | D
ax.axvline(5,  color='#444', linewidth=2.0)   # D | B
ax.axvline(11, color='#444', linewidth=2.0)   # B | C

# ── Horizontal separators between event groups ────────────────────────────────
prev_group = None
for i, event in enumerate(event_order):
    group = EVENT_GROUPS.get(event, 'Other')
    if prev_group is not None and group != prev_group:
        ax.axhline(i, color='#666', linewidth=1.2, linestyle='--', alpha=0.7)
    prev_group = group

# ── Two-part legend ───────────────────────────────────────────────────────────
col_legend_handles = [
    mpatches.Patch(color='midnightblue', label='Col A: Always present'),
    mpatches.Patch(color='darkgreen', label='Col B: Product-interaction'),
    mpatches.Patch(color='indigo', label='Col C: Purchase only'),
    mpatches.Patch(color='saddlebrown', label='Col D: Sparse / partial'),
]
evt_legend_handles = [
    mpatches.Patch(color=EGROUP_COLORS[g], label=f'Event: {g}')
    for g in GROUP_ORDER
]

leg1 = ax.legend(handles=col_legend_handles, title='Column groups',
                 title_fontsize=9, loc='upper left',
                 bbox_to_anchor=(0, -0.16), ncol=2, frameon=True, fontsize=9)
ax.add_artist(leg1)
ax.legend(handles=evt_legend_handles, title='Event groups',
          title_fontsize=9, loc='upper right',
          bbox_to_anchor=(1, -0.16), ncol=2, frameon=True, fontsize=9)

plt.tight_layout()
plt.show()


## 1.2. Temporal Patterns

Date range: **2025-06-01 to 2025-07-31** (61 days, Italy only).  
Timestamps are converted to **Europe/Rome** local time.

In [ ]:
ttp['event_dt']       = pd.to_datetime(ttp['event_timestamp'] / 1e6, unit='s', utc=True)
ttp['event_dt_local'] = ttp['event_dt'].dt.tz_convert('Europe/Rome')
ttp['hour']    = ttp['event_dt_local'].dt.hour
ttp['weekday'] = ttp['event_dt_local'].dt.day_name()
ttp['date']    = ttp['event_dt_local'].dt.date

WEEKDAY_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Daily event volume + purchase overlay
daily = ttp.groupby('date').size()
axes[0].plot(list(daily.index), daily.values, color='orange', linewidth=1.5)
axes[0].fill_between(list(daily.index), daily.values, alpha=0.2, color='orange')
axes[0].set_title('Daily Event Volume', fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Events')
axes[0].tick_params(axis='x', rotation=30, labelsize=8)
daily_purch = (ttp[ttp['event_name']=='purchase']
               .groupby('date').size().reindex(daily.index, fill_value=0))
ax2 = axes[0].twinx()
ax2.plot(list(daily_purch.index), daily_purch.values,
         color='royalblue', linewidth=1.5, linestyle='--', alpha=0.9, label='purchases')
ax2.set_ylabel('Purchase events', color='royalblue')
ax2.tick_params(axis='y', labelcolor='royalblue')
ax2.legend(loc='upper right', fontsize=8)


# Weekday x hour heatmap
pivot = ttp.pivot_table(index='weekday', columns='hour', values='session_id',
                        aggfunc='count').reindex(WEEKDAY_ORDER)
sns.heatmap(pivot, ax=axes[1], cmap='PuOr_r', linewidths=0.3,
            cbar_kws={'label': 'Events', 'shrink': 0.7})
axes[1].set_title('Events: Weekday x Hour', fontweight='bold')
axes[1].set_xlabel('Hour (Rome time)')
axes[1].set_ylabel('')

fig.suptitle('Temporal Patterns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 1.3. Device Source

In [ ]:
def infer_brand(model: str) -> str:
    if pd.isna(model):
        return 'Desktop'  # NaN = desktop, not a real unknown
    m = model.upper().strip()
    if m in ('IPHONE',) or 'IPHONE' in m:
        return 'Apple'
    elif m.startswith('SM-') or m.startswith('GT-') or m.startswith('SCG') or m.startswith('SC-'):
        return 'Samsung'
    elif re.match(r'^M\d{4}', m) or m.startswith('POCO') or m.startswith('MI '):
        return 'Xiaomi'
    elif m.startswith('CPH') or m.startswith('RMP') or m.startswith('PCRM'):
        return 'OPPO'
    elif m.startswith('V') and re.match(r'^V\d{4}', m):
        return 'Vivo'
    elif m.startswith('RNE') or m.startswith('ANE') or m.startswith('CLT') or m.startswith('ELS'):
        return 'Huawei'
    elif m.startswith('LM-') or m.startswith('LG-'):
        return 'LG'
    elif m.startswith('PIXEL') or m.startswith('GP') or m.startswith('GFE'):
        return 'Google'
    else:
        return 'Other'  # unrecognized mobile model
    
ttp['device_brand'] = ttp['device_model'].apply(infer_brand)

brand_summary = (
    ttp.groupby('device_brand')['device_model']
    .count()
    .sort_values(ascending=False)
    .reset_index(name='session_id')
)
brand_summary['share_%'] = (brand_summary['session_id'] / len(ttp) * 100).round(1)


In [ ]:
# Prepare data
ttp_plot = ttp.copy()
purchased_sessions = ttp_plot[ttp_plot['event_name'] == 'purchase']['session_id'].unique()
ttp_plot['purchased'] = ttp_plot['session_id'].isin(purchased_sessions).astype(int)

dev_purchase = ttp_plot.groupby(['device_category', 'purchased']).size().unstack(fill_value=0)
dev_purchase.columns = ['Non Purchase', 'Purchase']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Device category

dev_purchase.columns = ['Not Purchased', 'Purchased']
dev_purchase.plot(kind='bar', ax=axes[0],
                  color=['orange', 'royalblue'],
                  edgecolor='white', width=0.7)
axes[0].set_title('Device Category', fontweight='bold')
axes[0].set_ylabel('Sessions')
axes[0].tick_params(axis='x', rotation = 0)
axes[0].legend()

# --- Device Brand: Simple frequency bar (no purchase split) ---
brand_counts = (ttp_plot[ttp_plot['device_brand'] != 'Desktop']
                .groupby('device_brand')['session_id']
                .nunique()
                .sort_values(ascending=False))

# Device brand

brand_counts = (ttp_plot[ttp_plot['device_brand'] != 'Desktop']
                .groupby('device_brand')['session_id']
                .nunique()
                .sort_values(ascending=False))

axes[1].bar(brand_counts.index, brand_counts.values,
            color=['mediumslateblue', 'mediumseagreen', 'sandybrown', 'plum', 'lightcoral', 'khaki'],
            edgecolor='white')
axes[1].set_title('Device Brand', fontweight='bold')
axes[1].set_ylabel('Sessions')  # updated label
for bar, val in zip(axes[1].patches, brand_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{val:,}\n({val/brand_counts.sum()*100:.1f}%)',  # % of mobile sessions, not all events
                 ha='center', va='bottom', fontsize=9)

## 1.4. UTM

In [ ]:
def get_utm(url, param):
    try:
        return parse_qs(urlparse(url).query).get(param, [None])[0]
    except Exception:
        return None

ttp['utm_flag']    = ttp['page_location'].str.contains('utm_', na=False)
ttp['utm_source'] = ttp['page_location'].apply(lambda u: get_utm(u, 'utm_source'))
ttp['utm_medium'] = ttp['page_location'].apply(lambda u: get_utm(u, 'utm_medium'))

ttp['utm_source'] = ttp['utm_source'].str.lower()
ttp['utm_medium'] = ttp['utm_medium'].str.lower()

session_paid = ttp.groupby('session_id')['utm_flag'].any()

top_sources = (
    ttp[ttp['event_name'] == 'session_start']['utm_source']
    .value_counts().head(10)
)

top_mediums = (
    ttp[ttp['event_name'] == 'session_start']['utm_medium']
    .str.lower()
    .value_counts().head(8)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# --- Paid vs Organic pie ---
axes[0].pie(
    [session_paid.sum(), (~session_paid).sum()],
    labels=['Paid', 'Organic'],
    autopct='%1.1f%%',
    colors=['forestgreen', 'darkblue'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'color': 'white'}
)
axes[0].set_title('Sessions: Paid vs Organic', fontweight='bold')

# --- Top UTM sources bar ---
top_sources.plot(kind='barh', ax=axes[1], color='lightsteelblue', edgecolor='white')
axes[1].invert_yaxis()
axes[1].set_xlabel('Sessions')
axes[1].set_title('Top UTM Sources (session_start) \n (among paid only)', fontweight='bold')
axes[1].bar_label(
    axes[1].containers[0],
    labels=[f'{v/top_sources.sum()*100:.1f}%' for v in top_sources.values],
    padding=4
)
axes[1].spines[['top', 'right']].set_visible(False)

# --- Top UTM mediums bar ---
top_mediums.plot(kind='barh', ax=axes[2], color='royalblue', edgecolor='white')
axes[2].invert_yaxis()
axes[2].set_xlabel('Sessions')
axes[2].set_title('Top UTM Mediums (session_start) \n (among paid only)', fontweight='bold')
axes[2].bar_label(
    axes[2].containers[0],
    labels=[f'{v/top_mediums.sum()*100:.1f}%' for v in top_mediums.values],
    padding=4, color='black'
)
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

# Example paid URL
paid_ex = ttp[ttp['utm_flag']]['page_location'].dropna().iloc[0]
print(f"Example paid URL: {paid_ex[:120]}...")

### Campaign Analysis

Parsing `utm_campaign` to understand which specific campaigns drive traffic, classify them by type (sale, newsletter, paid search, social), and compare conversion rates across channels.


In [ ]:
# Parse utm_campaign
ttp['utm_campaign'] = ttp['page_location'].apply(lambda u: get_utm(u, 'utm_campaign'))

**Channel Classification**

We derive a `channel` label for each session from two UTM parameters extracted from `page_location`:
- **`utm_source`**: *who* sent the traffic (e.g. `google`, `instagram`, `sfmc`)
- **`utm_medium`**: *how* it was delivered (e.g. `cpc`, `email`, `social`)

When both are null, the user arrived with no tracking tags — meaning they typed the URL directly, used a bookmark, or came from an untracked link. We label these **organic**.

Otherwise, 
- we classify using **`utm_medium` as the primary signal**, since it follows a more controlled vocabulary
- we only fall back to `utm_source` when medium is uninformative or absent.

| Channel | What it means | Primary signal (medium) | Fallback (source) |
|---|---|---|---|
| `organic` | No paid or tracked campaign - direct/bookmarked visit | `medium = organic` | — |
| `email` | User clicked a link in a marketing or transactional email | medium contains `mail` | source is<br>- `sfmc` (Salesforce Marketing Cloud),<br>- `newsletter`, or<br>- `transactional` |
| `paid_search` | User clicked a<br>- paid Google or <br>- Bing search ad | medium is<br> - `cpc` (Cost Per Click),<br> - `sem` (Search Engine Marketing), or<br> - `ppc` (Pay Per Click) | source is `google` or `bing` |
| `shopping` | User clicked a Google Shopping product tile in search results | medium is `shopping` | — |
| `social` | User came from a social media platform | medium is `social` | source is `facebook` or `instagram` |
| `affiliate` | User clicked a link on a partner site tracked via Rakuten* | medium is `affiliate` | source is `rakuten` |
| `other` | UTM tags present but no pattern matched | — | — |

<br>
* Rakuten: an affiliate network that pays third parties a commission on referred sales


**Campaign Type Classification**

Beyond the channel, the `utm_campaign` string tells us the *marketing objective* behind the visit — i.e. what Versace was trying to achieve with that specific campaign. This gives us a second dimension to analyse traffic quality and intent.

We map campaign names to five types:

| Campaign type | What it means | Keyword signals |
|---|---|---|
| `organic` | No campaign tag present — user was not reached by any tracked campaign | null |
| `sale` | Promotional campaign tied to a discount or price event | `sale`, `markdown` |
| `newsletter` | Campaign sent via email newsletter | `mail`, `newsletter` |
| `paid_search` | Keyword-based search ad, often targeting branded or category terms | `search`, `text.`, `brand` |
| `retargeting` | "Follow-you" ads shown to users who already visited the site — served via Dynamic Product Ads (DPA) or Criteo, a specialist retargeting platform | `dpa`, `retarget`, `criteo` |
| `shopping` | Google Shopping product listing campaigns | `shopping`, `link.shopping` |
| `other` | Campaign tag exists but doesn't match any known pattern | — |

<br>

**Note:** channel and campaign type are complementary. For example, a `paid_search` channel session could be part of a `retargeting` campaign — the channel tells you *where* the ad appeared, the campaign type tells you *why* it was shown.

In [ ]:
# Classify channel from source + medium
def classify_channel(source, medium):
    if pd.isna(source) and pd.isna(medium):
        return 'organic'
    src = str(source).lower()
    med = str(medium).lower()
    
    # Medium is the primary signal
    if 'mail' in med:                          return 'email'
    if med in ('cpc', 'sem', 'ppc'):           return 'paid_search'
    if med == 'shopping':                      return 'shopping'
    if med == 'social':                        return 'social'
    if med == 'affiliate':                     return 'affiliate'
    if med == 'organic':                       return 'organic'
    
    # Fall back to source only when medium is uninformative
    if 'sfmc' in src or 'newsletter' in src or 'transactional' in src:
        return 'email'
    if 'google' in src or 'bing' in src:       return 'paid_search'
    if 'facebook' in src or 'instagram' in src: return 'social'
    if 'rakuten' in src:                       return 'affiliate'
    
    return 'other'

In [ ]:
# Classify campaign type from campaign name
def classify_campaign_type(camp):
    if pd.isna(camp):
        return 'organic'
    c = str(camp).lower()
    if 'sale' in c or 'markdown' in c:
        return 'sale'
    if 'mail' in c or 'newsletter' in c:
        return 'newsletter'
    if 'search' in c or 'text.' in c or 'brand' in c:
        return 'paid_search'
    if 'dpa' in c or 'retarget' in c or 'criteo' in c:
        return 'retargeting'
    if 'shopping' in c or 'link.shopping' in c:
        return 'shopping'
    return 'other'

In [ ]:
# Gender targeting from campaign name
def campaign_gender(camp):
    if pd.isna(camp): return 'unknown'
    c = str(camp).upper()
    if 'UOMO' in c: return 'men'
    if 'DONNA' in c: return 'women'
    return 'unspecified'

In [ ]:
ttp['channel'] = ttp.apply(
    lambda r: classify_channel(r['utm_source'], r['utm_medium']), axis=1)

ttp['campaign_type'] = ttp['utm_campaign'].apply(classify_campaign_type)

In [ ]:
ttp['campaign_gender'] = ttp['utm_campaign'].apply(campaign_gender)

# Session-level view (session_start events as anchor)
sess_start = ttp[ttp['event_name'] == 'session_start'].copy()


In [ ]:
# Shared color palette — same category = same color in both plots
COLOR_MAP = {
    'organic':     'forestgreen',   
    'paid_search': 'blue',   
    'email':       'darkorange',   
    'newsletter':  'orange',   # (email family)
    'shopping':    'purple',   
    'social':      'red',   
    'affiliate':   'gold',   
    'retargeting': 'teal',   
    'sale':        'pink',   
    'other':       'grey',   
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sessions by channel
ch_counts = sess_start['channel'].value_counts()
ch_colors = [COLOR_MAP.get(c, 'grey') for c in ch_counts.index]
ch_counts.plot(kind='barh', ax=axes[0], color=ch_colors, edgecolor='white')
axes[0].invert_yaxis()
axes[0].set_title('Sessions by Channel', fontweight='bold')
axes[0].set_xlabel('Session count (session_start events)')
axes[0].bar_label(
    axes[0].containers[0],
    labels=[f'{v:,} ({v/ch_counts.sum()*100:.1f}%)' for v in ch_counts.values],
    padding=4, fontsize=8
)
axes[0].spines[['top', 'right']].set_visible(False)

# Right: sessions by campaign type
camp_counts = sess_start['campaign_type'].value_counts()
camp_colors = [COLOR_MAP.get(c, '#95A5A6') for c in camp_counts.index]
camp_counts.plot(kind='barh', ax=axes[1], color=camp_colors, edgecolor='white')
axes[1].invert_yaxis()
axes[1].set_title('Sessions by Campaign Type', fontweight='bold')
axes[1].set_xlabel('Session count')
axes[1].bar_label(
    axes[1].containers[0],
    labels=[f'{v:,} ({v/camp_counts.sum()*100:.1f}%)' for v in camp_counts.values],
    padding=4, fontsize=8
)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('Traffic Distribution by Channel and Campaign Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def shorten_campaign(camp):
    if pd.isna(camp): return None
    c = str(camp)
    cl = c.lower()

    # Filter out transactional/payment noise
    if any(x in cl for x in ['klarna', 'orderconf', 'link.footer', 'link.order', '.dxc']):
        return None

    # Gender suffix helper
    gender = ' (Men)' if 'UOMO' in c.upper() else (' (Women)' if 'DONNA' in c.upper() else '')

    # Google Shopping
    if 'link.shopping' in cl: return 'Google Shopping'
    # Facebook DPA
    if 'dpa' in cl: return 'Facebook DPA'
    # Criteo retargeting
    if 'criteo' in cl: return 'Criteo Retargeting'
    # Brand search
    if 'top_brand_exact' in cl: return 'Brand Search: Exact'
    if 'brand_related' in cl:   return 'Brand Search: Related'
    # Paid search by category
    m = re.search(r'Search_([A-Za-z]+)_IT', c)
    if m: return f'Search: {m.group(1).title()}'

    # Ready-to-Wear — expand RTW acronym
    if 'rtw' in cl:
        if 'uomo' in cl or '_m' in cl:   return 'Ready-to-Wear (Men)'
        if 'donna' in cl or '_w' in cl:  return 'Ready-to-Wear (Women)'
        return 'Ready-to-Wear'

    # Newsletter / markdown — clean up truncated labels
    if 'mail' in cl or 'newsletter' in cl:
        if 'markdown' in cl:
            g = gender if gender else ''
            return f'Newsletter: Markdown{g}'
        m = re.search(r'mail_\d+\.\d+_?(\d+)?([A-Z][A-Z0-9_]+)', c)
        if m:
            label = m.group(2).replace('_', ' ').strip()
            return f'Newsletter: {label[:20]}{gender}'
        return f'Newsletter{gender}'

    # Sale campaigns — SS25/AW25 Public/Private Sale
    m = re.search(r'(SS\d+|AW\d+)[_-]?(Public|Private)?[_-]?Sale', c, re.IGNORECASE)
    if m:
        season   = m.group(1).upper()
        saletype = m.group(2).title() if m.group(2) else 'Public'
        geo      = ' Italy' if 'WM_IT' in c.upper() or '_IT' in c.upper() else ''
        g        = gender if gender else (' (Unspecified)' if not gender else '')
        return f'{season} {saletype} Sale{geo}{gender}'

    # Season + campaign name generic fallback
    m = re.search(r'(SS\d+|AW\d+)[_-]([A-Za-z0-9]+)', c)
    if m:
        label = m.group(0).replace('_', ' ').replace('-', ' ')
        return f'{label}{gender}'

    # Last resort: first 35 chars, replace underscores
    return c[:35].replace('_', ' ')

# ← apply shortening BEFORE groupby to avoid duplicate labels
sess_start['campaign_label'] = sess_start['utm_campaign'].apply(shorten_campaign)

top_camps = (
    sess_start[sess_start['campaign_label'].notna()]
    .groupby('campaign_label')['session_id'].count()
    .sort_values(ascending=False)
    .head(15)
)

gender_counts = (
    ttp[ttp['utm_campaign'].notna()]['campaign_gender']
    .value_counts()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_camps.plot(kind='barh', ax=axes[0], color=bar_colors, edgecolor='white')
axes[0].invert_yaxis()
axes[0].set_title('Top 15 Campaigns by Session Volume', fontweight='bold')
axes[0].set_xlabel('Sessions')
axes[0].bar_label(axes[0].containers[0], padding=4, fontsize=8)
axes[0].legend(handles=legend_handles, title='Campaign Type',
               fontsize=8, title_fontsize=8, loc='lower right')
axes[0].set_ylabel('')
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].pie(
    gender_counts.values,
    labels=gender_counts.index,
    autopct='%1.1f%%',
    colors=['#888888', '#A8C8F0', '#F4A7B9'],
    hatch = ['', '///' ,'---'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

plt.suptitle('Campaign Volume and Targeting', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
top_camps = (
    sess_start[sess_start['campaign_label'].notna()]
    .groupby(['campaign_label', 'campaign_type'])['session_id'].count()
    .reset_index()
    .sort_values('session_id', ascending=False)
    .head(15)
    .set_index('campaign_label')
)

bar_colors = [COLOR_MAP.get(t, '#95A5A6') for t in top_camps['campaign_type']]

# Legend
legend_types = top_camps['campaign_type'].unique()
legend_handles = [Patch(facecolor=COLOR_MAP.get(t, '#95A5A6'), label=t)
                  for t in sorted(legend_types)]

In [ ]:
# Shared segment helper
def top_seg(cat):
    if pd.isna(cat): return 'other'
    c = str(cat).lower()
    for seg in ['women', 'men', 'children', 'home', 'jeans']:
        if c.startswith(seg): return 'jeans couture' if seg == 'jeans' else seg
    return 'other'

seg_colors = {'men': '#1E88E5', 'women': '#E91E63', 'children': '#43A047',
              'home': '#FF9800', 'jeans couture': '#9C27B0', 'other': '#78909C'}

view_items = ttp[ttp['event_name'] == 'view_item'].copy()
view_items = view_items[view_items['item_price'].notna() & (view_items['item_price'] > 0)]
view_items['segment'] = view_items['item_category'].apply(top_seg)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# ── COMPUTE SHARED ORDERS ONCE ───────────────────────────────────────────────
CHANNELS   = ['email', 'paid_search', 'shopping', 'social', 'organic', 'affiliate']
CAMP_TYPES = ['sale', 'newsletter', 'paid_search', 'shopping', 'retargeting', 'organic']

channel_order = (view_items[view_items['channel'].isin(CHANNELS)]
                 .groupby('channel')['item_price']
                 .median().sort_values(ascending=False)
                 .reindex([c for c in CHANNELS if c in view_items['channel'].unique()])
                 .sort_values(ascending=False).index.tolist())

camp_order = (view_items[view_items['campaign_type'].isin(CAMP_TYPES)]
              .groupby('campaign_type')['item_price']
              .median().sort_values(ascending=False).index.tolist())

# ── ROW 1: PRICE BOX PLOTS ───────────────────────────────────────────────────

# [0,0] Box plot by channel — uses channel_order
data_by_channel = [view_items[view_items['channel'] == ch]['item_price'].values
                   for ch in channel_order]
bp1 = axes[0,0].boxplot(data_by_channel, labels=channel_order, patch_artist=True,
                         showfliers=False,
                         medianprops={'color': 'white', 'linewidth': 2})
for patch, ch in zip(bp1['boxes'], channel_order):
    patch.set_facecolor(COLOR_MAP.get(ch, '#95A5A6'))
axes[0,0].set_title('Item Price by Channel (IQR, outliers hidden)', fontweight='bold')
axes[0,0].set_ylabel('Item price (EUR)')
axes[0,0].tick_params(axis='x')
axes[0,0].spines[['top', 'right']].set_visible(False)

# [0,1] Box plot by campaign type — uses camp_order
data_by_camp = [view_items[view_items['campaign_type'] == ct]['item_price'].values
                for ct in camp_order]
bp2 = axes[0,1].boxplot(data_by_camp, labels=camp_order, patch_artist=True,
                         showfliers=False,
                         medianprops={'color': 'white', 'linewidth': 2})
for patch, ct in zip(bp2['boxes'], camp_order):
    patch.set_facecolor(COLOR_MAP.get(ct, '#95A5A6'))
axes[0,1].set_title('Item Price by Campaign Type (IQR, outliers hidden)', fontweight='bold')
axes[0,1].set_ylabel('Item price (EUR)')
axes[0,1].tick_params(axis='x')
axes[0,1].spines[['top', 'right']].set_visible(False)

# ── ROW 2: CATEGORY MIX — reuses same orders ──────────────────────────────────
SEGMENTS = ['women', 'men', 'home', 'children', 'jeans couture', 'other']
pastel_colors = {
    'women':        '#F4A7B9',
    'men':          '#A8C8F0',
    'home':         '#FFDCA8',
    'children':     '#A8E6B0',
    'jeans couture':'#D4AEEA',
    'other':        '#C8C8C8',
}
hatches = {
    'women':        '---',
    'men':          '///',
    'home':         '...',
    'children':     'xxx',
    'jeans couture':'\\\\\\',
    'other':        '',
}

def plot_category_mix(ax, view_df, group_col, group_order, title):
    pivot = (
        view_df[view_df[group_col].isin(group_order)]
        .groupby([group_col, 'segment']).size().unstack(fill_value=0)
    )
    for seg in SEGMENTS:
        if seg not in pivot.columns:
            pivot[seg] = 0
    pivot = pivot[SEGMENTS]
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    # ← key fix: reindex using the same order as the boxplot above
    pivot_pct = pivot_pct.reindex(index=[c for c in group_order if c in pivot_pct.index])

    bottom = pd.Series([0.0] * len(pivot_pct), index=pivot_pct.index)
    for seg in SEGMENTS:
        vals = pivot_pct[seg]
        ax.bar(pivot_pct.index, vals, bottom=bottom,
               color=pastel_colors[seg], hatch=hatches[seg],
               edgecolor='white', linewidth=0.5, label=seg)
        bottom += vals

    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Share (%)')
    ax.tick_params(axis='x')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(loc='upper right', fontsize=8, bbox_to_anchor=(1.18, 1))

# pass the same order variables used in the boxplots above
plot_category_mix(axes[1,0], view_items, 'channel',       channel_order,
                  'Item Category Mix by Channel (%)')
plot_category_mix(axes[1,1], view_items, 'campaign_type', camp_order,
                  'Item Category Mix by Campaign Type (%)')

plt.suptitle('Item Browsing Patterns by Channel and Campaign Type',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 1.5. User Journey

In [ ]:
first_visit = (
    ttp.groupby('user_pseudo_id')['event_date']
       .min()
       .rename('first_visit')
)

purchase_date = (
    ttp[ttp['event_name'] == 'purchase']
       .groupby('user_pseudo_id')['event_date']
       .min()
       .rename('purchase_date')
)

journey = pd.concat([first_visit, purchase_date], axis=1)
journey['days_to_purchase'] = (
    pd.to_datetime(journey['purchase_date']) - pd.to_datetime(journey['first_visit'])
).dt.days

print(journey['days_to_purchase'].describe())
print(f"\nBought same day (day 0): {(journey['days_to_purchase'] == 0).mean():.1%}")

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.35)
axes = [[fig.add_subplot(gs[r, c]) for c in range(3)] for r in range(2)]

# ── pre-compute ───────────────────────────────────────────────────────────────
sessions_per_user = (
    ttp.drop_duplicates(['user_pseudo_id', 'session_id'])
       .groupby('user_pseudo_id')['session_id'].count()
)
cap_spu = int(sessions_per_user.quantile(0.98))

events_per_session = ttp.groupby('session_id').size()
cap_eps = int(events_per_session.quantile(0.95))

eng_per_session = ttp.groupby('session_id')['engagement_time_msec'].sum().div(1000)
cap_eng = int(eng_per_session.quantile(0.95))

session_meta = (
    ttp.groupby('session_id')
       .agg(
           eng_sec          = ('engagement_time_msec', lambda x: x.sum() / 1000),
           events           = ('event_name',           'count'),
           is_purchase_sess = ('event_name',           lambda x: (x == 'purchase').any().astype(int)),
           session_n        = ('ga_session_number',    'first'),
       )
)

purchase_sn = ttp[ttp['event_name'] == 'purchase']['ga_session_number']
cap_psn     = int(purchase_sn.quantile(0.95))
pct_first   = (purchase_sn == 1).mean() * 100

first_visit   = ttp.groupby('user_pseudo_id')['event_date'].min()
purchase_date = (ttp[ttp['event_name'] == 'purchase']
                   .groupby('user_pseudo_id')['event_date'].min())
journey = pd.concat([first_visit.rename('first'), purchase_date.rename('purchase')], axis=1)
journey['days'] = (pd.to_datetime(journey['purchase']) - pd.to_datetime(journey['first'])).dt.days
day0_pct = (journey['days'] == 0).mean() * 100
later     = journey[journey['days'] > 0]['days']
cap_days  = int(later.quantile(0.95))

# ── helpers ───────────────────────────────────────────────────────────────────
BROWSE_COL  = 'orange'
CONVERT_COL = 'mediumpurple'

def add_lines(ax, series):
    med, mu = series.median(), series.mean()
    ax.axvline(med, color='red',   linestyle='--', linewidth=1.5, label=f'Median={med:.1f}')
    ax.axvline(mu,  color='black', linestyle=':',  linewidth=1.5, label=f'Mean={mu:.1f}')
    ax.legend(fontsize=8)

def kde_bounded(ax, data, cap, color, label):
    x   = np.linspace(0, cap, 500)
    kde = gaussian_kde(data.clip(lower=0, upper=cap), bw_method='scott')
    y   = np.clip(kde(x), 0, None)
    ax.plot(x, y, color=color, linewidth=1.5, label=label)
    ax.fill_between(x, y, alpha=0.35, color=color)
    ax.axvline(data.median(), color=color, linestyle='--', linewidth=1.2,
               label=f'median={data.median():.0f}')

# [0,0]  Sessions per user
axes[0][0].hist(sessions_per_user.clip(upper=cap_spu),
                bins=range(1, cap_spu + 2), color='midnightblue', edgecolor='white', align='left')
axes[0][0].set_xlabel(f'Sessions per user (capped at p98={cap_spu})')
axes[0][0].set_ylabel('Users')
axes[0][0].set_title('Sessions per User')
add_lines(axes[0][0], sessions_per_user)

# [0,1]  Events per session — browsing vs converting
for label, mask, color in [
    ('Browsing',   session_meta['is_purchase_sess'] == 0, BROWSE_COL),
    ('Converting', session_meta['is_purchase_sess'] == 1, CONVERT_COL),
]:
    kde_bounded(axes[0][1],
                session_meta.loc[mask, 'events'],
                cap_eps, color, label)
axes[0][1].set_xlim(0, cap_eps)
axes[0][1].set_xlabel(f'Events per session (capped at p95={cap_eps})')
axes[0][1].set_ylabel('Density')
axes[0][1].set_title('Events per Session')
axes[0][1].legend(fontsize=8)

# [0,2]  Engagement time — browsing vs converting
for label, mask, color in [
    ('Browsing',   session_meta['is_purchase_sess'] == 0, BROWSE_COL),
    ('Converting', session_meta['is_purchase_sess'] == 1, CONVERT_COL),
]:
    kde_bounded(axes[0][2],
                session_meta.loc[mask, 'eng_sec'],
                cap_eng, color, label)
axes[0][2].set_xlim(0, cap_eng)
axes[0][2].set_xlabel(f'Engagement time / session, sec (capped at p95={cap_eng})')
axes[0][2].set_ylabel('Density')
axes[0][2].set_title('Engagement Time per Session')
axes[0][2].legend(fontsize=8)


# [1,0]  Dual-axis: events + engagement by visit number
visit_stats = (
    session_meta[session_meta['session_n'] <= 15]
    .groupby('session_n')
    .agg(med_eng=('eng_sec', 'median'), med_ev=('events', 'median'))
)
ax_ev  = axes[1][0]
ax_eng = ax_ev.twinx()

bars = ax_ev.bar(visit_stats.index, visit_stats['med_ev'],
                 color='darkcyan', alpha=0.6, edgecolor='white', label='Events (left)')
ax_eng.plot(visit_stats.index, visit_stats['med_eng'],
            color='palevioletred', marker='o', linewidth=1.8, markersize=4, label='Engagement (right)')

ax_ev.set_xlabel('Visit number (sessions 1–15)')
ax_ev.set_ylabel('Median events per session', color='darkcyan')
ax_eng.set_ylabel('Median engagement time (sec)', color='palevioletred')
ax_ev.tick_params(axis='y', labelcolor='darkcyan')
ax_eng.tick_params(axis='y', labelcolor='palevioletred')
ax_ev.set_xticks(range(1, 16))
ax_ev.set_title('Events & Engagement by Visit Number')
lines1, labels1 = ax_ev.get_legend_handles_labels()
lines2, labels2 = ax_eng.get_legend_handles_labels()
ax_ev.legend(lines1 + lines2, labels1 + labels2, fontsize=7)
ax_ev.grid(False)

# [1,1]  Which visit = first purchase
axes[1][1].hist(purchase_sn.clip(upper=cap_psn),
                bins=range(1, cap_psn + 2), color='mediumpurple', edgecolor='white', align='left')
axes[1][1].set_xlabel(f'Visit number at purchase (capped at p95={cap_psn})')
axes[1][1].set_ylabel('Purchases')
axes[1][1].set_title('Which Visit Did Users Buy On?')
axes[1][1].text(0.52, 0.88, f'{pct_first:.1f}% bought\non 1st visit',
                transform=axes[1][1].transAxes, fontsize=9,
                bbox=dict(boxstyle='round', facecolor='mediumpurple', alpha=0.3))
add_lines(axes[1][1], purchase_sn)

# [1,2]  Days from first visit to purchase
bins = [-0.5, 0.5] + list(range(1, cap_days + 2))
n, bins_out, patches = axes[1][2].hist(
    journey['days'].clip(upper=cap_days), bins=bins,
    color='slategray', edgecolor='white'
)
patches[0].set_facecolor('#E74C3C')
axes[1][2].text(0, n[0] + 8, f'{day0_pct:.1f}%\nsame day',
                ha='center', fontsize=9, color='#E74C3C', fontweight='bold')
axes[1][2].axvline(later.median(), color='red',   linestyle='--', linewidth=1.5,
                   label=f'Median (day 1+) = {later.median():.0f}d')
axes[1][2].axvline(later.mean(),   color='black', linestyle=':',  linewidth=1.5,
                   label=f'Mean (day 1+) = {later.mean():.1f}d')
axes[1][2].set_xlabel(f'Days from first visit to purchase (capped at p95={cap_days})')
axes[1][2].set_ylabel('Users')
axes[1][2].set_title('Days to Purchase')
axes[1][2].legend(fontsize=8)

# column headers 
col_themes = ['WHO  ·  user volume', 'HOW DEEP  ·  session intensity', 'WHEN  ·  purchase timing']
for col, theme in enumerate(col_themes):
    axes[0][col].set_title(f'{theme}\n{axes[0][col].get_title()}', fontsize=9)

fig.suptitle('TTP  –  User Journey Structure & Engagement Depth', fontsize=14)
plt.show()

In [ ]:
# ga_session_number: does it reset at dataset start, or is it a lifetime counter?
# If users enter the dataset already at session 5+, our 'first view' is NOT their real first view.
user_min_session = ttp.groupby('user_pseudo_id')['ga_session_number'].min()

fresh = (user_min_session == 1).sum()
mid   = (user_min_session > 1).sum()
total = len(user_min_session)

print(f'Users whose first record is session 1  (new):         {fresh:,}  ({fresh/total*100:.1f}%)')
print(f'Users whose first record is session >1 (mid-journey): {mid:,}  ({mid/total*100:.1f}%)')
print(f'Max ga_session_number at first appearance: {user_min_session.max():.0f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: new vs mid-journey
axes[0].bar(
    ['Enters at session 1\n(new to dataset)', 'Enters at session >1\n(mid-journey)'],
    [fresh, mid], color=['#43A047', '#E53935'], edgecolor='white'
)
axes[0].set_ylabel('Number of users')
axes[0].set_title('Are Users New or Mid-Journey at Dataset Start?', fontsize=12, fontweight='bold')
for bar, val, pct in zip(axes[0].patches, [fresh, mid], [fresh/total*100, mid/total*100]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 0.5,
                 f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold', color='white')

# Right: distribution of min session number (capped at 20+)
capped = user_min_session.clip(upper=23)
counts = capped.value_counts().sort_index()
bar_colors = ['#43A047' if i == 1 else '#E53935' for i in counts.index]
axes[1].bar([str(int(x)) if x < 30 else '30+' for x in counts.index],
            counts.values, color=bar_colors, edgecolor='white')
axes[1].set_xlabel('First ga_session_number seen in dataset')
axes[1].set_ylabel('Number of users')
axes[1].set_title('Distribution of Session Number at First Appearance', fontsize=12, fontweight='bold')

fig.suptitle('Left Truncation Check — ga_session_number is a Lifetime Counter',
             fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

print('\nImplication for survival analysis:')
print(f'  {mid} users ({mid/total*100:.1f}%) entered the dataset mid-journey.')


Their first view in our data is NOT their true first exposure. Duration estimates for these users are left-truncated (underestimated).

## 1.6. Item categories

In [ ]:
CHILD_SUBSEG_RE = re.compile(
    r'(boy|girl|baby|infant|kid|toddler|\d{1,2}[-–]\d{1,2}\s*y)',
    re.IGNORECASE
)

def parse_category(cat):
    if not isinstance(cat, str) or cat in ('(not set)', ''):
        return pd.Series({
            'brand': None, 'segment': None, 'sub_segment': None,
            'product_type': None, 'sub_type': None, 'leaf_category': None
        })

    p = [x.strip() for x in cat.split(':')]
    get = lambda i: p[i] if len(p) > i else None  # safe index helper

    if p[0] == 'jeans couture':
        brand, seg, sub_seg = 'Versace Jeans Couture', get(1), None
        pt, st = get(2), get(3)

    elif p[0] == 'home and lifestyle':
        brand, seg, sub_seg = 'Versace', 'home & lifestyle', None
        pt, st = get(1), get(2)

    elif p[0] in ('men', 'women'):
        brand, seg, sub_seg = 'Versace', p[0], None
        pt, st = get(1), get(2)

    elif p[0] == 'children':
        brand, seg = 'Versace', 'children'
        if len(p) > 1 and CHILD_SUBSEG_RE.search(p[1]):
            sub_seg, pt, st = p[1], get(2), get(3)
        else:
            sub_seg, pt, st = None, get(1), get(2)

    else:  # no-gender prefix: bags, sunglasses, …
        brand, seg, sub_seg = 'Versace', None, None
        pt, st = p[0], get(1)

    # leaf_category: the most specific comparable level across all segments
    # men/women/VJC go one level deeper (clothing → denim) so use sub_type there
    if seg in ('men', 'women') or brand == 'Versace Jeans Couture':
        leaf = st if st is not None else pt
    else:
        leaf = pt  # children (sub_segment is age bracket, not a category), home & lifestyle, no-gender

    return pd.Series({
        'brand': brand, 'segment': seg, 'sub_segment': sub_seg,
        'product_type': pt, 'sub_type': st, 'leaf_category': leaf
    })


# ── Apply ─────────────────────────────────────────────────────────────────────
ttp_items = ttp[ttp['item_category'].notna() & (ttp['item_category'] != '(not set)')].copy()
ttp_items = pd.concat([ttp_items, ttp_items['item_category'].apply(parse_category)], axis=1)

In [ ]:
ttp_items[['segment', 'sub_segment', 'product_type', 'sub_type', 'leaf_category']].head(10)

In [ ]:
import matplotlib.gridspec as gridspec

In [ ]:
fashion_segs = ['men', 'women', 'children']

# ── Prep: session-level purchase flag ────────────────────────────────────────
session_purchased = (
    ttp[ttp['event_name'] == 'purchase'][['session_id']]
    .drop_duplicates()
    .assign(purchased=1)
)
ttp_items = ttp_items.merge(session_purchased, on='session_id', how='left')
ttp_items['purchased'] = ttp_items['purchased'].fillna(0).astype(int)

# ── Prep: session × leaf_category deduplicated ───────────────────────────────
session_leaf = (
    ttp_items[
        ttp_items['segment'].isin(fashion_segs) &
        ttp_items['leaf_category'].notna()
    ]
    [['session_id', 'segment', 'leaf_category', 'purchased']]
    .drop_duplicates(['session_id', 'leaf_category'])
)

In [ ]:
# ── Prep ──────────────────────────────────────────────────────────────────────
seg_order = ['men', 'women', 'children', 'home & lifestyle']

session_brand_seg = (
    ttp_items[ttp_items['segment'].notna()]
    [['session_id', 'brand', 'segment', 'purchased']]
    .drop_duplicates(['session_id', 'segment'])
)

counts = (
    session_brand_seg.groupby(['segment', 'brand', 'purchased'])
    .agg(sessions=('session_id', 'nunique'))
    .reset_index()
)

cvr_seg = (
    session_brand_seg.groupby(['segment', 'brand'])
    .agg(sessions=('session_id', 'nunique'), purchases=('purchased', 'sum'))
    .assign(cvr=lambda d: d['purchases'] / d['sessions'] * 100)
    .reset_index()
)

In [ ]:
brands        = ['Versace', 'Versace Jeans Couture']

In [ ]:
COLORS = {
    'Versace':               {'non_purchase': '#E8735A', 'purchase': '#4878CF'},
    'Versace Jeans Couture': {'non_purchase': '#f4b8ae', 'purchase': '#a8c4e8'},  # pastel versions
}
CVR_COLOR = '#2a9d8f'  # teal

fig, ax1 = plt.subplots(figsize=(13, 6))

x      = np.arange(len(seg_order))
width  = 0.35
offset = {'Versace': -width/2, 'Versace Jeans Couture': width/2}

for brand in brands:
    not_purch = (
        counts[(counts['brand'] == brand) & (counts['purchased'] == 0)]
        .set_index('segment')['sessions']
        .reindex(seg_order, fill_value=0)
    )
    purch = (
        counts[(counts['brand'] == brand) & (counts['purchased'] == 1)]
        .set_index('segment')['sessions']
        .reindex(seg_order, fill_value=0)
    )
    off = offset[brand]
    ax1.bar(x + off, not_purch, width,
            color=COLORS[brand]['non_purchase'], edgecolor='white',
            label=f'{brand} — Non-Purchase')
    ax1.bar(x + off, purch, width,
            bottom=not_purch, color=COLORS[brand]['purchase'], edgecolor='white',
            label=f'{brand} — Purchase')

ax1.set_ylabel('Sessions')
ax1.set_xticks(x)
ax1.set_xticklabels(seg_order, fontsize=10)
ax1.spines[['top']].set_visible(False)
ax1.legend(loc='upper right', fontsize=8, ncol=2)

ax2 = ax1.twinx()
for brand in brands:
    cvr_vals = (
        cvr_seg[cvr_seg['brand'] == brand]
        .set_index('segment')['cvr']
        .reindex(seg_order)
    )
    shade = CVR_COLOR if brand == 'Versace' else '#76c8c0'  # lighter teal for VJC
    ax2.plot(x + offset[brand], cvr_vals,
             color=shade, marker='o', linewidth=2,
             markersize=6, linestyle='--',
             label=f'{brand} CVR %')

ax2.set_ylabel('CVR %', color=CVR_COLOR)
ax2.tick_params(axis='y', labelcolor=CVR_COLOR)
ax2.spines[['top']].set_visible(False)
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(False)

plt.title('Sessions & Conversion Rate by Segment × Brand')
plt.tight_layout()
plt.show()

In [ ]:
TOP_N = 20  # ← change only this

# ── Prep ──────────────────────────────────────────────────────────────────────
top_n = (
    session_leaf.groupby('leaf_category')['session_id']
    .nunique().nlargest(TOP_N).index
)
session_leaf_n = session_leaf[session_leaf['leaf_category'].isin(top_n)]
wrapped = [l.replace(' ', '\n') for l in top_n]

# Mask: True where a segment × leaf_category combo is structurally unavailable
# (i.e. never appears anywhere in the raw data, not just zero in purchased/not)
available = (
    session_leaf_n.groupby(['segment', 'leaf_category'])
    .size().unstack(fill_value=0)
    .reindex(fashion_segs)
    .reindex(columns=top_n, fill_value=0)
)
na_mask = available == 0  # True = gray out

def make_heatmap(data, flag, ax, cmap, title):
    hm = (
        data[data['purchased'] == flag]
        .groupby(['segment', 'leaf_category']).size()
        .unstack(fill_value=0)
        .reindex(fashion_segs)
        .reindex(columns=top_n, fill_value=0)
    )
    hm.columns = wrapped
    mask = na_mask.copy()
    mask.columns = wrapped

    sns.heatmap(hm, annot=True, fmt=',', cmap=cmap, mask=mask,
                linewidths=0.5, ax=ax, cbar_kws={'label': 'sessions'})
    # Gray overlay for masked cells
    sns.heatmap(pd.DataFrame(mask.values, columns=wrapped, index=fashion_segs),
                mask=~mask.values, cmap=['#d3d3d3'], alpha=0.8,
                linewidths=0.5, ax=ax, cbar=False, annot=False)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=8, rotation=0)
    ax.tick_params(axis='y', rotation=0)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

make_heatmap(session_leaf_n, 0, axes[0], 'Reds',  'Non-Purchase Sessions')
make_heatmap(session_leaf_n, 1, axes[1], 'Blues', 'Purchase Sessions')

cvr_hm = (
    session_leaf_n.groupby(['segment', 'leaf_category'])
    .agg(sessions=('session_id', 'nunique'), purchases=('purchased', 'sum'))
    .assign(cvr=lambda d: d['purchases'] / d['sessions'] * 100)
    ['cvr']
    .unstack(fill_value=0)
    .reindex(fashion_segs)
    .reindex(columns=top_n, fill_value=0)
)
cvr_mask = na_mask.copy()
cvr_hm.columns = wrapped
cvr_mask.columns = wrapped

sns.heatmap(cvr_hm, annot=True, fmt='.1f', cmap='coolwarm_r', mask=cvr_mask,
            linewidths=0.5, ax=axes[2], cbar_kws={'label': 'CVR %'})
sns.heatmap(pd.DataFrame(cvr_mask.values, columns=wrapped, index=fashion_segs),
            mask=~cvr_mask.values, cmap=['#d3d3d3'], alpha=0.8,
            linewidths=0.5, ax=axes[2], cbar=False, annot=False)
axes[2].set_title('Conversion Rate %', fontsize=12)
axes[2].tick_params(axis='x', labelsize=8, rotation=0)
axes[2].tick_params(axis='y', rotation=0)

plt.suptitle(f'Category Browsing vs. Purchase Behavior (Top {TOP_N} Categories)')
plt.tight_layout()
plt.show()

In [ ]:
TOP_N_HL = 15  # ← change only this

# ── Prep ──────────────────────────────────────────────────────────────────────
session_leaf_hl = (
    ttp_items[
        (ttp_items['segment'] == 'home & lifestyle') &
        ttp_items['leaf_category'].notna()
    ]
    [['session_id', 'leaf_category', 'purchased']]
    .drop_duplicates(['session_id', 'leaf_category'])
)

top_n_hl = (
    session_leaf_hl.groupby('leaf_category')['session_id']
    .nunique().nlargest(TOP_N_HL).index
)
session_leaf_hl = session_leaf_hl[session_leaf_hl['leaf_category'].isin(top_n_hl)]

cvr_hl = (
    session_leaf_hl.groupby('leaf_category')
    .agg(sessions=('session_id', 'nunique'), purchases=('purchased', 'sum'))
    .assign(cvr=lambda d: d['purchases'] / d['sessions'] * 100)
    .reindex(top_n_hl)
)

not_purchased = (
    session_leaf_hl[session_leaf_hl['purchased'] == 0]
    .groupby('leaf_category')['session_id'].nunique()
    .reindex(top_n_hl, fill_value=0)
)
purchased = (
    session_leaf_hl[session_leaf_hl['purchased'] == 1]
    .groupby('leaf_category')['session_id'].nunique()
    .reindex(top_n_hl, fill_value=0)
)

# Sort by total sessions for consistent ordering across all 3 panels
order = cvr_hl['sessions'].sort_values().index
wrapped_hl = [l.replace(' ', '\n') for l in order]
x = range(len(order))

fig, ax1 = plt.subplots(figsize=(10, 5))

categories = order  # the 4 product types
x = np.arange(len(categories))
width = 0.35

ax1.bar(x - width/2, not_purchased.reindex(order), width,
        label='Non-Purchase', color='lightsalmon', edgecolor='white')
ax1.bar(x + width/2, purchased.reindex(order), width,
        label='Purchase', color='lightsteelblue', edgecolor='white')
ax1.set_ylabel('Sessions')
ax1.set_xticks(x)
ax1.set_xticklabels([l.replace(' ', '\n') for l in order], fontsize=9)
ax1.spines[['top']].set_visible(False)
ax1.legend()

# CVR line on secondary axis
ax2 = ax1.twinx()
ax2.plot(x, cvr_hl.reindex(order)['cvr'], color='teal',
         marker='o', linewidth=2, markersize=6, label='CVR %', linestyle = '--')
ax2.set_ylabel('CVR %', color='teal')
ax2.tick_params(axis='y', labelcolor='teal')
ax2.spines[['top']].set_visible(False)
ax2.legend()
ax2.grid(False)

plt.title('Home & Lifestyle — Sessions & Conversion Rate by Category')
plt.tight_layout()
plt.show()

In [ ]:
import textwrap

In [ ]:
# ── Shared prep ───────────────────────────────────────────────────────────────
viewed    = ttp[ttp['event_name'] == 'view_item'].copy()
viewed['viewed_value'] = viewed['item_price'] * viewed['item_quantity'].fillna(1)
purchased = ttp[ttp['event_name'] == 'purchase'].copy()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Versace TTP — Revenue Leakage Analysis', fontsize=16, fontweight='bold')


# ANGLE 2 — Revenue Realization Rate by Category
ax = axes[0, 0]
view_by_cat  = viewed.groupby('item_category')['viewed_value'].sum().rename('viewed')
purch_by_cat = purchased.groupby('item_category')['item_revenue'].sum().rename('realized')
cat_df = pd.concat([view_by_cat, purch_by_cat], axis=1).dropna().reset_index()
cat_df['realization_rate'] = cat_df['realized'] / cat_df['viewed'] * 100
cat_df = cat_df.sort_values('viewed', ascending=False).head(12)

x = np.arange(len(cat_df))
w = 0.38
ax.bar(x - w/2, cat_df['viewed'] / 1e3,   width=w, label='Viewed (€k)',   color='#E8604C', alpha=0.85)
ax.bar(x + w/2, cat_df['realized'] / 1e3, width=w, label='Realized (€k)', color='#4C72B0', alpha=0.85)
ax2 = ax.twinx()
ax2.plot(x, cat_df['realization_rate'], color='black', marker='o', linewidth=2, markersize=5, label='Realization %')
ax2.set_ylabel('Realization Rate %')
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
#  replace : and space with \n in the tick labels for better readability
xlabels = [l.replace(':', '\n').replace(' ', '\n') for l in cat_df['item_category']]

ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=8)
ax.set_ylabel('Value (€k)')
ax.set_title('Revenue Realization Rate by Category', fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
ax.grid(False)

# ANGLE 3 — Pareto
ax = axes[0, 1]
user_rev = (
    purchased.groupby('user_pseudo_id')['item_revenue']
    .sum().sort_values(ascending=False).reset_index()
)
user_rev['cum_rev_pct']  = user_rev['item_revenue'].cumsum() / user_rev['item_revenue'].sum() * 100
user_rev['cum_user_pct'] = np.arange(1, len(user_rev) + 1) / len(user_rev) * 100

idx_80 = (user_rev['cum_rev_pct'] >= 80).idxmax()
pct_users_for_80 = user_rev.loc[idx_80, 'cum_user_pct']

ax.plot(user_rev['cum_user_pct'], user_rev['cum_rev_pct'], color='#4C72B0', linewidth=2.5)
ax.fill_between(user_rev['cum_user_pct'], user_rev['cum_rev_pct'], alpha=0.12, color='#4C72B0')
ax.axhline(80, color='#E8604C', linestyle='--', linewidth=1.4)
ax.axvline(pct_users_for_80, color='#E8604C', linestyle='--', linewidth=1.4)
ax.annotate(
    f"Top {pct_users_for_80:.1f}% of users\ngenerate 80% of revenue",
    xy=(pct_users_for_80, 80), xytext=(pct_users_for_80 + 8, 55),
    arrowprops=dict(arrowstyle='->', color='#E8604C'), fontsize=9, color='#E8604C'
)
ax.plot([0, 100], [0, 100], 'k--', linewidth=1, alpha=0.3, label='Perfect equality')
ax.set_xlabel('Cumulative % of Users')
ax.set_ylabel('Cumulative % of Revenue')
ax.set_title('Revenue Concentration (Pareto)', fontweight='bold')
ax.set_xlim(0, 100); ax.set_ylim(0, 100)
ax.set_aspect('equal')
ax.legend(fontsize=8)

# ANGLE 4 — Cart Value Trajectory
ax = axes[1, 0]
buyers = purchased['user_pseudo_id'].unique()
cart_trajectory = (
    ttp[ttp['user_pseudo_id'].isin(buyers)]
    .dropna(subset=['cart_value_estimate'])
    .groupby(['user_pseudo_id', 'ga_session_number'])['cart_value_estimate']
    .max().reset_index()
)
purchase_session = purchased.groupby('user_pseudo_id')['ga_session_number'].max().rename('purchase_session_no')
cart_trajectory  = cart_trajectory.merge(purchase_session, on='user_pseudo_id')
cart_trajectory['sessions_before_purchase'] = (
    cart_trajectory['purchase_session_no'] - cart_trajectory['ga_session_number']
)
traj_agg = (
    cart_trajectory[cart_trajectory['sessions_before_purchase'].between(0, 10)]
    .groupby('sessions_before_purchase')['cart_value_estimate']
    .agg(['mean', 'median', 'count']).reset_index()
    .sort_values('sessions_before_purchase', ascending=False)
)
xlabels = [f"N-{int(s)}" if s > 0 else "Purchase\nSession" for s in traj_agg['sessions_before_purchase']]
ax.plot(xlabels, traj_agg['mean'],   marker='o', color='goldenrod', linewidth=2.5, markersize=7, label='Mean')
ax.plot(xlabels, traj_agg['median'], marker='s', color='#55A868', linewidth=2,   markersize=6, linestyle='--', label='Median')
ax.fill_between(xlabels, traj_agg['mean'], alpha=0.1, color='goldenrod')
for i, row in enumerate(traj_agg.itertuples()):
    ax.text(i, row.mean + 15, f"n={int(row.count)}", ha='center', fontsize=7, color='gray')
ax.set_xlabel('Sessions Relative to Purchase')
ax.set_ylabel('Cart Value Estimate (€)')
ax.set_title('Cart Value Trajectory Before Purchase', fontweight='bold')
ax.tick_params(axis='x', labelsize=8)
ax.legend(fontsize=8)

# ANGLE 5 — Browsed vs Purchased Price per User (all sessions)
ax = axes[1, 1]

# User-level avg browsed price (all view_item events, all sessions)
user_browsed = (
    ttp[ttp['event_name'] == 'view_item']
    .groupby('user_pseudo_id')['item_price']
    .mean()
    .rename('browsed')
)
# User-level avg purchased price (all purchase events)
user_purchased = (
    purchased.groupby('user_pseudo_id')['item_price']
    .mean()
    .rename('purch')
)
# Only users present in both
price_compare = pd.concat([user_browsed, user_purchased], axis=1).dropna()

cap = 1500
v = price_compare['browsed'].clip(upper=cap).values
p = price_compare['purch'].clip(upper=cap).values

x_range = np.linspace(0, cap, 500)
kde_v = gaussian_kde(v, bw_method=0.3)
kde_p = gaussian_kde(p, bw_method=0.3)

ax.plot(x_range, kde_v(x_range), color='#E8604C', linewidth=2.5, label='Avg Browsed Price')
ax.plot(x_range, kde_p(x_range), color='#4C72B0', linewidth=2.5, label='Avg Purchased Price')
ax.fill_between(x_range, kde_v(x_range), alpha=0.15, color='#E8604C')
ax.fill_between(x_range, kde_p(x_range), alpha=0.15, color='#4C72B0')
ax.axvline(np.median(v), color='#E8604C', linestyle='--', linewidth=1.4,
           label=f'Browsed median: €{np.median(v):.0f}')
ax.axvline(np.median(p), color='#4C72B0', linestyle='--', linewidth=1.4,
           label=f'Purchased median: €{np.median(p):.0f}')
ax.set_xlabel('Item Price (€)')
ax.set_ylabel('Density')
ax.set_title('Browsed vs Purchased Price Distribution\n(User-Level Avg, All Sessions)', fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# 2. Feature Engineering

All features are built once here and reused by both models.

| Dataset | Unit | Used by |
|---------|------|---------|
| `journeys` | one row per (user, item) pair | Survival Analysis (Part 2) |
| `session_df` | one row per session | Binary Session Model (Part 3) |

## 2.1. Journey perspective


**Unit of analysis:** one row per (user, item) pair.

**Duration:** hours from the user's *first `view_item` event* for that item to either:
- the *purchase event* for that item (event = 1), or
- the *last recorded event* for that user (event = 0, right-censored)

**Why right-censored?** Users who viewed an item but never bought it within the
observation window may still purchase later -- we just don't observe it.
We cannot treat them as non-converters without inflating failure rates.

### Build `journey` dataframe

In [ ]:
# Product-level rows only (item_id present and not placeholder)
prod = ttp[ttp['item_id'].notna() & (ttp['item_id'] != '(not set)')].copy()

In [ ]:
# First view_item event per (user, item)
first_view = (
    prod[prod['event_name'] == 'view_item']
    .sort_values('event_dt')
    .groupby(['user_pseudo_id', 'item_id'])
    .agg(
        first_view_dt          = ('event_dt',          'min'),
        device_category        = ('device_category',   'first'),
        login_status           = ('login_status',      'first'),
        ga_session_num_at_view = ('ga_session_number', 'min'),
        item_price             = ('item_price',        'first'),
        item_category          = ('item_category',     'first'),
    )
    .reset_index()
)

In [ ]:
# First purchase per (user, item)
purchase_dt = (
    prod[prod['event_name'] == 'purchase']
    .groupby(['user_pseudo_id', 'item_id'])['event_dt']
    .min()
    .rename('purchase_dt')
    .reset_index())

# Join
journeys = first_view.merge(purchase_dt, 
                            on=['user_pseudo_id', 'item_id'], 
                            how='left')

journeys['purchased'] = journeys['purchase_dt'].notna().astype(int)

In [ ]:
# Censoring: last observed event per user
last_event_dt = (ttp.groupby('user_pseudo_id')['event_dt']
                 .max()
                 .rename('last_event_dt')
                 .reset_index())

journeys = journeys.merge(last_event_dt, on='user_pseudo_id')

In [ ]:
# Duration
journeys['end_dt']         = journeys['purchase_dt'].fillna(journeys['last_event_dt'])
journeys['duration_hours'] = (journeys['end_dt'] - journeys['first_view_dt']).dt.total_seconds() / 3600
journeys['duration_days']  = journeys['duration_hours'] / 24

# Drop zero/negative durations (same-microsecond events -- data artifact)
journeys = journeys[journeys['duration_hours'] > 0].copy()

In [ ]:
# Journey table overview
overview = pd.DataFrame({
    'Metric': ['Total journey pairs', 'Unique users', 'Unique items',
               'Purchased (event=1)', 'Censored (event=0)'],
    'Value': [
        f'{len(journeys):,}',
        f'{journeys["user_pseudo_id"].nunique():,}',
        f'{journeys["item_id"].nunique():,}',
        f'{journeys["purchased"].sum():,}',
        f'{(1 - journeys["purchased"]).sum():,}',
    ]
})
display(overview)
print()
print('Duration (hours) by outcome:')
display(journeys.groupby('purchased')['duration_hours'].describe().round(1))

In [ ]:
# --- Repeat purchases: same user bought same item more than once --------------
repeat = (
    prod[prod['event_name'] == 'purchase']
    .groupby(['user_pseudo_id', 'item_id'])['event_dt']
    .count()
    .rename('n_purchases')
)
n_repeat    = (repeat > 1).sum()
n_purchased = journeys['purchased'].sum()
n_total     = len(journeys)
print(f'Purchased pairs:                          {n_purchased:,}')
print(f'  of which purchased more than once:      {n_repeat:,} ({n_repeat/n_purchased*100:.1f}% of purchased pairs)')
print(f'  as share of ALL pairs (incl. censored): {n_repeat/n_total*100:.1f}%')

Decision: keep only first purchase (already done via `.min()` on `purchase_dt`). -- too few to model separately.

### Set up variables


**Item segment**

In [ ]:
# Build lookup to recover missing categories from other rows of the same item_id

category_lookup = (
    prod[prod['item_category'].notna() & (prod['item_category'] != '(not set)')]
    .groupby('item_id')['item_category'].first()
)

In [ ]:
def parse_segment(row):
    cat     = row['item_category']
    item_id = row['item_id']
    if pd.isna(cat) or cat == '(not set)':
        cat = category_lookup.get(item_id, None)
        if cat is None:
            return None
    l0 = str(cat).split(':')[0].strip().lower()
    if l0 == 'men':                return 'men'
    if l0 == 'women':              return 'women'
    if l0 == 'children':           return 'children'
    if l0 == 'home and lifestyle': return 'home and lifestyle'
    if l0 == 'jeans couture':      return 'jeans couture'
    return 'other'  # bags, sunglasses -- too small for own coefficient

In [ ]:
journeys['segment'] = journeys.apply(parse_segment, axis=1)

seg_counts = journeys['segment'].value_counts(dropna=False).rename('count').to_frame()
seg_counts['%'] = (seg_counts['count'] / len(journeys) * 100).round(1)
display(seg_counts)
print(f'Unrecoverable (excluded from Cox): {journeys["segment"].isna().sum():,}')

In [ ]:
seg_dummies = pd.get_dummies(journeys['segment'], prefix='seg', drop_first=False)
seg_dummies = seg_dummies.drop(columns=['seg_men'], errors='ignore')  # men = reference
journeys = pd.concat([journeys, seg_dummies], axis=1)
seg_cols = [c for c in journeys.columns if c.startswith('seg_')]

**Login status and device**

In [ ]:
journey_login = (                                                                                                                                                                                    prod[prod['event_name'].isin(['view_item', 'purchase'])]
      .merge(journeys[['user_pseudo_id', 'item_id']], 
             on=['user_pseudo_id', 'item_id'])                                                                                                                .groupby(['user_pseudo_id', 'item_id'])['login_status']
      .nunique()
      .rename('n_distinct')
  )
n_mixed = (journey_login > 1).sum()
print(f'User-item journeys with mixed login_status: {n_mixed:,} / {len(journey_login):,} ({n_mixed/len(journey_login)*100:.1f}%)')

In [ ]:
journeys['is_logged_in'] = (journeys['login_status'] == 'loggedin').astype(int)
journeys['is_mobile']    = (journeys['device_category'] == 'mobile').astype(int)

**Loyalty signal**

Using log-transformed continuous value rather than a binary flag: captures both 
- the loyalty signal (more sessions = more brand familiarity) and 
- the truncation severity (how much unobserved history exists).
Log transform because of extreme skew (max session ~878).

In [ ]:
# Log-transform: captures loyalty signal + handles skew (max session ~878)
journeys['log_ga_session_num'] = np.log1p(journeys['ga_session_num_at_view'].fillna(1))

In [ ]:
# Has prior purchase (repeat buyer signal)
user_first_purchase = (
    prod[prod['event_name'] == 'purchase']
    .groupby('user_pseudo_id')['event_dt'].min()
    .rename('first_purchase_dt')
)
journeys = journeys.join(user_first_purchase, on='user_pseudo_id')
journeys['has_prior_purchase'] = (
    journeys['first_purchase_dt'] < journeys['first_view_dt']
).astype(int)
journeys.drop(columns=['first_purchase_dt'], inplace=True)

**Campaign**

Attribution is at the **(user, item)** level: we take the channel from the session in which the user first viewed that specific item — not their very first session ever. 

This answers *which channel surfaced this item to this user*, which is the right signal for the Cox model (predicting time-to-purchase of a specific item). 

User-acquisition channel would be appropriate for a user-level model, not a journey-level one.

In [ ]:
# --- Campaign type at first view of each item (Cox UTM feature) -----------
def get_utm_campaign(url):
    try:
        return parse_qs(urlparse(url).query).get('utm_campaign', [None])[0]
    except:
        return None

def classify_campaign_type(camp):
    if pd.isna(camp): return 'organic'
    c = str(camp).lower()
    if 'sale' in c or 'markdown' in c:                 return 'sale'
    if 'mail' in c or 'newsletter' in c:               return 'newsletter'
    if 'search' in c or 'text.' in c or 'brand' in c:  return 'paid_search'
    if 'dpa' in c or 'retarget' in c or 'criteo' in c: return 'retargeting'
    if 'shopping' in c or 'link.shopping' in c:        return 'shopping'
    return 'other'

In [ ]:
# Step 1: get session_id of first view_item per (user, item)
first_view_session = (
    prod[prod['event_name'] == 'view_item']
    .sort_values('event_dt')
    .groupby(['user_pseudo_id', 'item_id'])['session_id']
    .first()
    .reset_index()
)

# Step 2: get campaign type from session_start of that session
# UTM tags appear on session_start URLs, not product page URLs
sess_campaign = (
    ttp[ttp['event_name'] == 'session_start'][['session_id', 'page_location']]
    .drop_duplicates('session_id')
)
sess_campaign['utm_campaign']  = sess_campaign['page_location'].apply(get_utm_campaign)
sess_campaign['campaign_type'] = sess_campaign['utm_campaign'].apply(classify_campaign_type)

# Step 3: join campaign type onto journeys via session_id
first_view_camp = first_view_session.merge(
    sess_campaign[['session_id', 'campaign_type']], 
    on='session_id', how='left'
)
first_view_camp['campaign_type'] = first_view_camp['campaign_type'].fillna('organic')

# Step 4: dummies -- organic is reference category
camp_dummies = pd.get_dummies(first_view_camp['campaign_type'], prefix='camp').astype(int)
camp_dummies = camp_dummies.drop(columns=['camp_organic'], errors='ignore')
first_view_camp = pd.concat([first_view_camp[['user_pseudo_id', 'item_id']], camp_dummies], axis=1)

journeys   = journeys.merge(first_view_camp, on=['user_pseudo_id', 'item_id'], how='left')
camp_cols  = [c for c in journeys.columns if c.startswith('camp_')]
journeys[camp_cols] = journeys[camp_cols].fillna(0).astype(int)

print('Campaign type distribution:')
print(journeys[camp_cols].sum().sort_values(ascending=False))

**Price**

In [ ]:
price_median = journeys['item_price'].median()
journeys['item_price']   = journeys['item_price'].fillna(price_median)
journeys['log_price']    = np.log1p(journeys['item_price'])

**Time**

In [ ]:
# Hour of first view
journeys['hour_of_day'] = journeys['first_view_dt'].dt.hour

In [ ]:
eng_at_view = (
    prod[prod['event_name'] == 'view_item']
    .sort_values('event_dt')
    .groupby(['user_pseudo_id', 'item_id'])['engagement_time_msec']
    .first().reset_index()
    .rename(columns={'engagement_time_msec': 'engage_time_ms'})
)

In [ ]:
journeys = journeys.merge(eng_at_view, on=['user_pseudo_id', 'item_id'], how='left')
journeys['engage_time_ms']  = journeys['engage_time_ms'].fillna(0)
journeys['log_engage_time'] = np.log1p(journeys['engage_time_ms'])

**Summary**

In [ ]:
# Journey feature summary
feat_cols = (['log_price', 'is_mobile', 'log_ga_session_num', 'has_prior_purchase',
              'log_engage_time'] + camp_cols + seg_cols)
display(journeys[feat_cols].describe().round(3))
print('Baseline segment for Cox: men')

## 2.2. Binary session

### Set up variables

In [ ]:
# Load raw events, coerce numerics, add session_key
df = ttp.copy()
for col in ['ga_session_number', 'engagement_time_msec', 'item_price', 'cart_value_estimate']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['session_key'] = df['user_pseudo_id'].astype(str) + '_' + df['session_id'].astype(str)
df = df.sort_values(['user_pseudo_id', 'event_dt']).reset_index(drop=True)

**Campaign**

In [ ]:
# Define UTM parsing helpers
def _has_utm(url):
    return 0 if pd.isna(url) else int('utm_' in str(url).lower())

def _get_utm(url, param):
    if pd.isna(url): return np.nan
    try:
        return parse_qs(urlparse(str(url)).query).get(param, [np.nan])[0]
    except Exception:
        return np.nan

def safe_mode(x): # null-safe version of "most frequent value"
    x = x.dropna()
    return x.mode().iloc[0] if len(x) > 0 else np.nan

In [ ]:
# Apply helpers (UTM flags)
df['utm_flag_row']     = df['page_location'].apply(_has_utm)
df['utm_source_row']   = df['page_location'].apply(lambda u: _get_utm(u, 'utm_source'))
df['utm_campaign_row'] = df['page_location'].apply(lambda u: _get_utm(u, 'utm_campaign'))

**Time**

In [ ]:
# Add other time features (hour, weekday) 
df['event_hour']       = df['event_dt'].dt.hour
df['event_weekday']    = df['event_dt'].dt.day_name()

**Event-type dummies**

In [ ]:
# Step 3/5: Add 
df['is_purchase']      = (df['event_name'] == 'purchase').astype(int)
df['is_view_item']     = (df['event_name'] == 'view_item').astype(int)
df['is_add_to_cart']   = (df['event_name'] == 'add_to_cart').astype(int)

### Aggregating to build Session dataframe

Since the goal is to predict purchase behavior at the session level, the data was aggregated so that each row represents a single session.

Event-level rows are collapsed into one row per session using three types of aggregation:

- **"Grab any value"** (first, max, safe_mode): for columns that are constant within a session, like device, login status, UTM source, and hour/weekday.
- **"Did it happen?"** (max on 0/1 flags): for binary outcomes like purchase_in_session and utm_flag. If any event triggered it, the session gets a 1.
- **"How much / how many?"** (sum, mean, max, nunique): for behavioral intensity: event counts, price exposure, item breadth, engagement time.

In [ ]:
session_df = (
    df.groupby('session_key')
      .agg(
          user_pseudo_id           = ('user_pseudo_id',        'first'),
          session_id               = ('session_id',            'first'),
          ga_session_number        = ('ga_session_number',     'max'),
          session_start            = ('event_dt',              'min'),
          session_end              = ('event_dt',              'max'),
          purchase_in_session      = ('is_purchase',           'max'),
          device_category          = ('device_category',       safe_mode),
          login_status             = ('login_status',          safe_mode),
          utm_flag                 = ('utm_flag_row',          'max'),
          utm_source               = ('utm_source_row',        safe_mode),
          utm_campaign             = ('utm_campaign_row',      safe_mode),
          first_hour               = ('event_hour',            'first'),
          weekday                  = ('event_weekday',         'first'),
          n_events                 = ('event_name',            'size'),
          n_unique_items           = ('item_id',               pd.Series.nunique),
          n_view_item              = ('is_view_item',          'sum'),
          n_add_to_cart            = ('is_add_to_cart',        'sum'),
          avg_item_price           = ('item_price',            'mean'),
          max_item_price           = ('item_price',            'max'),
          avg_engagement_time_msec = ('engagement_time_msec',  'mean'),
          main_item_category       = ('item_category',         safe_mode)
      )
      .reset_index()
)

print(f'Sessions: {len(session_df):,} ')

In [ ]:
session_df['session_duration_sec'] = (
    session_df['session_end'] - session_df['session_start']
).dt.total_seconds()

session_df['add_to_view_ratio'] = (
    session_df['n_add_to_cart'] / session_df['n_view_item'].replace(0, np.nan)
)

In [ ]:
# NaN utm means organic/direct; impute before pipeline to avoid mislabelling
session_df['utm_source']   = session_df['utm_source'].fillna('direct')
session_df['utm_campaign'] = session_df['utm_campaign'].fillna('direct')

### Adding user-history features (loyalty, recency)

In [ ]:
session_df = session_df.sort_values(['user_pseudo_id', 'session_start']).reset_index(drop=True)

session_df['prev_sessions']  = session_df.groupby('user_pseudo_id').cumcount()
session_df['prev_purchases'] = (
    session_df.groupby('user_pseudo_id')['purchase_in_session']
              .transform(lambda x: x.cumsum().shift(fill_value=0))
)
session_df['days_since_prev_session'] = (
    session_df.groupby('user_pseudo_id')['session_start']
              .diff().dt.total_seconds() / 86400
)
session_df['has_purchased_before'] = (session_df['prev_purchases'] > 0).astype(int)

hist_summary = pd.DataFrame({
    'Metric': ['Sessions with prior purchase', 'Max prev_sessions', 'Median days since prev session'],
    'Value': [
        f'{session_df["has_purchased_before"].sum():,} ({session_df["has_purchased_before"].mean()*100:.1f}%)',
        str(int(session_df['prev_sessions'].max())),
        f'{session_df["days_since_prev_session"].median():.1f}'
    ]
})
display(hist_summary)

**Summary**

In [ ]:
# Session feature overview
sess_cols = ['n_events', 'n_unique_items', 'n_view_item', 'n_add_to_cart',
             'avg_item_price', 'session_duration_sec', 'add_to_view_ratio',
             'prev_sessions', 'has_purchased_before', 'purchase_in_session']
display(session_df[sess_cols].describe().round(3))

## 2.3 Feature Map: Cox vs Binary Session Model

Both models draw on the same signals but encode them differently.

Cox PH requires numeric/dummy inputs and log-transforms for proportional-hazards stability.

Tree models handle raw categoricals and non-linearity natively.

**Shared concepts -- different encoding:**

| Concept | Cox PH *(journey-level)* | Binary Session *(session-level)* | Why different form |
|---------|--------------------------|----------------------------------|--------------------|
| Device | `is_mobile` (binary) | `device_category` (raw) | Cox: numeric HR<br> Trees handle strings natively |
| Loyalty | `log_ga_session_num` at first view | `ga_session_number`<br>`prev_sessions` | Cox: log to reduce skew<br>session model: raw + separate recency counter |
| Purchase history | `has_prior_purchase` (stratified -- PH violation) | `has_purchased_before`<br>`prev_purchases` | Cox: stratification avoids biased HR<br> trees: learn non-linear interaction directly |
| Price | `log_price` (item-level, log) | `avg_item_price`<br>`max_item_price` | Cox: one item's conversion<br>session: aggregate across all items browsed |
| Channel / UTM | `camp_*` dummies (5 classified types) | `utm_flag`<br>`utm_source`<br>`utm_campaign` (raw) | Cox: interpretable HR per channel type<br>trees: free splits on raw values |
| Item category | `seg_*` dummies (6 parsed segments) | `main_item_category` (raw) | Cox needs dummies<br>OrdinalEncoder for trees |
| Engagement | `log_engage_time` (first-view, log) | `avg_engagement_time_msec`<br>`session_duration_sec` | Cox: depth of initial consideration<br>session: total activity across the session |


**Session-only:**

| Feature | Reason not in Cox |
|---------|-------------------|
| `login_status` | Session-stable (mode per session)<br>Cox drops it because login state can shift across a multi-day journey window |
| `n_events`<br>`n_view_item`<br>`n_add_to_cart`<br>`add_to_view_ratio` | Within-session event counts -- no equivalent at item-journey level (Cox sees one snapshot per item) |
| `weekday`<br>`first_hour` | Timing of the session -- not applicable at item-journey level |
| `days_since_prev_session` | Inter-session recency gap -- not a journey-level concept |

## 2.4 NLP Enrichment: Item Similarity Feature

The baseline model knows about individual items but not about the *browsing session as a whole*.
A user viewing 5 similar handbags has different intent than one browsing unrelated categories.

**`session_item_sim`** = average pairwise cosine similarity of OpenAI `text-embedding-3-small` embeddings
for all items viewed in the session. High value = focused, intent-driven browsing.

**Model:** OpenAI `text-embedding-3-small` (1536-dim) via API.
Embeddings are computed once, saved to `item_embeddings.pkl`, and loaded below.
API key is stored in `.env` and loaded with `python-dotenv` — never hard-coded in the notebook.


In [ ]:
# # ── RUN ONCE to generate item_embeddings.pkl (OpenAI text-embedding-3-small) ──────────────
# # Requires: OPENAI_API_KEY in .env  |  pip install openai python-dotenv
# # Uncomment and run manually once; result is saved and reloaded in the next cell.

# import os, time, pickle
# import numpy as np
# from dotenv import load_dotenv
# from openai import OpenAI

# load_dotenv()
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# mba = pd.read_csv("IMA2026_Versace_MBA_Product_Level.csv", low_memory=False)
# mba_item = (mba[["item_id","operative_brand","commercial_category",
#                   "commercial_sub_category","style_fabric_color"]]
#             .drop_duplicates("item_id"))
# ttp_item = (prod[prod["item_id"].notna() & (prod["item_id"] != "(not set)")]
#             [["item_id","item_category","item_price"]].drop_duplicates("item_id"))
# item_meta = ttp_item.merge(mba_item, on="item_id", how="left")

# def build_item_text(row):
#     parts = []
#     for col in ["operative_brand","commercial_category","commercial_sub_category","style_fabric_color"]:
#         v = row.get(col)
#         if pd.notna(v) and str(v).strip():
#             parts.append(str(v).strip())
#     cat = row.get("item_category")
#     if pd.notna(cat) and str(cat) not in ("(not set)", ""):
#         parts.append(str(cat).strip())
#     return " | ".join(parts) if parts else "unknown item"

# item_meta["item_text"] = item_meta.apply(build_item_text, axis=1)
# item_text_map = item_meta.set_index("item_id")["item_text"].to_dict()
# item_ids_list = list(item_text_map.keys())
# texts_list    = [item_text_map[i] for i in item_ids_list]

# def embed_batch(texts):
#     for attempt in range(5):
#         try:
#             response = client.embeddings.create(
#                 input=texts,
#                 model="text-embedding-3-small"
#             )
#             return [e.embedding for e in response.data]
#         except Exception as e:
#             wait = 60 if "429" in str(e) else 5
#             print(f"  Rate limited, waiting {wait}s (attempt {attempt+1}/5)...")
#             time.sleep(wait)
#     raise RuntimeError("embed_batch failed after 5 attempts")

# BATCH_SIZE = 50
# all_embeddings = []
# for start in range(0, len(texts_list), BATCH_SIZE):
#     batch = texts_list[start : start + BATCH_SIZE]
#     all_embeddings.extend(embed_batch(batch))
#     print(f"  {min(start + BATCH_SIZE, len(texts_list))}/{len(texts_list)} items embedded...")
#     time.sleep(1.0)

# embeddings_array = np.array(all_embeddings)
# norms = np.linalg.norm(embeddings_array, axis=1, keepdims=True)
# embeddings_array = embeddings_array / np.where(norms == 0, 1, norms)
# item_embeddings = dict(zip(item_ids_list, embeddings_array))

# with open("item_embeddings.pkl", "wb") as f:
#     pickle.dump(item_embeddings, f)
# print(f"Saved {len(item_embeddings):,} embeddings  (dim={len(all_embeddings[0])})")


In [ ]:
# Load pre-computed item embeddings
with open('item_embeddings.pkl', 'rb') as f:
    item_embeddings = pickle.load(f)

sample_id = next(iter(item_embeddings))
print(f'Loaded {len(item_embeddings):,} item embeddings  (dim={len(item_embeddings[sample_id])})')

In [ ]:
# Compute session-level item similarity (average pairwise cosine sim)
# L2-normalised embeddings: cosine sim = dot product
session_viewed = (
    prod[prod['event_name'] == 'view_item']
    .groupby('session_id')['item_id']
    .apply(lambda ids: [i for i in ids.unique() if i in item_embeddings])
    .reset_index().rename(columns={'item_id': 'viewed_items'})
)

def avg_pairwise_sim(items):
    if len(items) < 2: return np.nan
    vecs = np.stack([item_embeddings[i] for i in items])
    sim_matrix = vecs @ vecs.T
    n = len(items)
    return float(sim_matrix[np.triu_indices(n, k=1)].mean())

session_viewed['session_item_sim'] = session_viewed['viewed_items'].apply(avg_pairwise_sim)
med_sim = session_viewed['session_item_sim'].median()
session_viewed['session_item_sim'] = session_viewed['session_item_sim'].fillna(med_sim)

# Map session_item_sim onto journeys via first-view session_id
first_view_sess_ids = (
    prod[prod['event_name'] == 'view_item'].sort_values('event_dt')
    .groupby(['user_pseudo_id', 'item_id'])['session_id']
    .first().reset_index()
)
journeys = journeys.merge(first_view_sess_ids[['user_pseudo_id', 'item_id', 'session_id']],
                           on=['user_pseudo_id', 'item_id'], how='left')
journeys = journeys.merge(session_viewed[['session_id', 'session_item_sim']], on='session_id', how='left')
journeys['session_item_sim'] = journeys['session_item_sim'].fillna(med_sim)

display(pd.DataFrame({
    'Outcome': ['Not purchased (0)', 'Purchased (1)'],
    'Mean session_item_sim': journeys.groupby('purchased')['session_item_sim'].mean().round(4).values
}))

# 3. Survival Analysis

**Research question:** How long does it take from first viewing an item to purchasing it?
Which factors accelerate or delay conversion?

**Method:** Cox Proportional Hazards model on user-item journeys.

We assess 2 models: (i) baseline (no item similarity using LLM api) then (ii) enriched (+ item similarity) -- lets us
quantify the incremental value of session focus.

**Relationship to survival analysis:** The Cox model asks *how long* until purchase across
sessions. The binary session model asks *which sessions* result in a purchase -- a complementary,
session-level view that is actionable in real-time (trigger nudges for high-probability sessions).

In [ ]:
# Journey completeness: did we observe the user from session 1?
journeys['started_mid_journey'] = (journeys['ga_session_num_at_view'] > 1).astype(int)

full_mask = journeys['started_mid_journey'] == 0
part_mask = journeys['started_mid_journey'] == 1

completeness = pd.DataFrame({
    'Group': ['Full journey (from session 1)', 'Partial journey (joined mid-way)'],
    'N': [full_mask.sum(), part_mask.sum()],
    'Purchased': [journeys[full_mask]['purchased'].sum(), journeys[part_mask]['purchased'].sum()],
    'Conv %': [journeys[full_mask]['purchased'].mean()*100, journeys[part_mask]['purchased'].mean()*100]
}).assign(**{'Conv %': lambda d: d['Conv %'].round(2)})
display(completeness)

In [ ]:
# Journey completeness heatmap
ct = pd.crosstab(
    journeys['started_mid_journey'].map({0: 'Full journey', 1: 'Partial journey'}),
    journeys['purchased'].map({0: 'Did not purchase', 1: 'Purchased'})
)
ct_pct = ct / ct.values.sum() * 100

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.heatmap(ct_pct, annot=True, fmt='.1f', cmap='Blues', cbar_kws={'label': '% of total'}, ax=ax)
ax.set_title('Journey completeness x purchase outcome (% of all pairs)', fontsize=12, pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11, rotation=0)
plt.tight_layout()
plt.show()

## 3.1 Kaplan-Meier Survival Curves

The **survival function S(t)** = probability that the item has *not yet been purchased*
by time t after first view.

- S(t) starts at 1.0 (no purchase yet at t=0)
- At each time point where a purchase happens, the curve drops. 

    The formula for each step is:
    $$S(t_k) = S(t_{k-1}) \times (1  - \frac {d_k }{ n_k})$$
    where 
    - $d_k$ = number of purchases at time $t_k$, and 
    - $n_k$ = number of users still "at risk" (not yet purchased and not censored).
- The curve *never reaches 0* because most items are never purchased (censored)
- **Median time to purchase** = t where S(t) = 0.5

In [ ]:
kmf_all = KaplanMeierFitter()
kmf_all.fit(journeys['duration_days'], journeys['purchased'])
overall_sf = kmf_all.survival_function_

def add_overall(ax):
    ax.plot(overall_sf.index, overall_sf['KM_estimate'],
            color='#999', linewidth=2, linestyle='--', label='Overall', zorder=1)

fig, axes = plt.subplots(2, 4, figsize=(28, 12))

# R1C1: Device
ax = axes[0, 0]; add_overall(ax)
for device, (color, label) in {'mobile': ('#2196F3', 'Mobile'), 'desktop': ('goldenrod', 'Desktop')}.items():
    mask = journeys['device_category'] == device
    if mask.sum() < 10: continue
    kmf = KaplanMeierFitter()
    kmf.fit(journeys.loc[mask, 'duration_days'], journeys.loc[mask, 'purchased'], label=f'{label} (n={mask.sum():,})')
    kmf.plot_survival_function(ax=ax, color=color, linewidth=1.2, ci_show=True, ci_alpha=0.12)
mob  = journeys[journeys['device_category'] == 'mobile']
desk = journeys[journeys['device_category'] == 'desktop']
lr   = logrank_test(mob['duration_days'], desk['duration_days'], mob['purchased'], desk['purchased'])
ax.set_title('By Device Type', fontsize=11, fontweight='bold')
ax.set_xlabel(f'Days  (log-rank p={lr.p_value:.4f})'); ax.set_ylabel('S(t)'); ax.set_xlim(left=0); ax.legend(fontsize=8)

# R1C2: Login Status
ax = axes[0, 1]; add_overall(ax)
grp_d, grp_e = [], []
for flag, (color, label) in {1: ('#43A047', 'Logged in'), 0: ('black', 'Not logged in')}.items():
    mask = journeys['is_logged_in'] == flag
    kmf  = KaplanMeierFitter()
    kmf.fit(journeys.loc[mask, 'duration_days'], journeys.loc[mask, 'purchased'], label=f'{label} (n={mask.sum():,})')
    kmf.plot_survival_function(ax=ax, color=color, linewidth=1.2, ci_show=True, ci_alpha=0.12)
    grp_d.append(journeys.loc[mask, 'duration_days']); grp_e.append(journeys.loc[mask, 'purchased'])
lr = logrank_test(grp_d[0], grp_d[1], grp_e[0], grp_e[1])
ax.set_title('By Login Status', fontsize=11, fontweight='bold')
ax.set_xlabel(f'Days  (log-rank p={lr.p_value:.4f})'); ax.set_ylabel('S(t)'); ax.set_xlim(left=0); ax.legend(fontsize=8)

# R1C3: Campaign Type
ax = axes[0, 2]; add_overall(ax)
camp_km_map = {
    'camp_sale': ('#E53935', 'Sale'), 'camp_newsletter': ('#1E88E5', 'Newsletter'),
    'camp_paid_search': ('#43A047', 'Paid search'), 'camp_shopping': ('#FF9800', 'Shopping'),
    'camp_retargeting': ('#9C27B0', 'Retargeting'),
}
for col, (color, label) in camp_km_map.items():
    if col not in journeys.columns: continue
    mask = journeys[col] == 1
    if mask.sum() < 10: continue
    kmf = KaplanMeierFitter()
    kmf.fit(journeys.loc[mask, 'duration_days'], journeys.loc[mask, 'purchased'], label=f'{label} (n={mask.sum():,})')
    kmf.plot_survival_function(ax=ax, color=color, linewidth=1.2, ci_show=False)
ax.set_title('By Campaign Type', fontsize=11, fontweight='bold')
ax.set_xlabel('Days since first view'); ax.set_ylabel('S(t)'); ax.set_xlim(left=0); ax.legend(fontsize=7)

# R1C4: Item Segment
ax = axes[0, 3]; add_overall(ax)
seg_colors = {'men': '#1E88E5', 'women': '#E91E63', 'children': '#43A047',
              'home and lifestyle': '#FF9800', 'jeans couture': '#9C27B0', 'other': '#78909C'}
for seg, color in seg_colors.items():
    mask = journeys['segment'] == seg
    if mask.sum() < 10: continue
    kmf = KaplanMeierFitter()
    kmf.fit(journeys.loc[mask, 'duration_days'], journeys.loc[mask, 'purchased'], label=f'{seg} (n={mask.sum():,})')
    kmf.plot_survival_function(ax=ax, color=color, linewidth=1.2, ci_show=False)
ax.set_title('By Item Segment', fontsize=11, fontweight='bold')
ax.set_xlabel('Days since first view'); ax.set_ylabel('S(t)'); ax.set_xlim(left=0); ax.legend(fontsize=7)

# R2C1: Prior Purchase
ax = axes[1, 0]; add_overall(ax)
grp_d, grp_e = [], []
for flag, (color, label) in {1: ('#E53935', 'Prior buyer'), 0: ('#1E88E5', 'First-time viewer')}.items():
    mask = journeys['has_prior_purchase'] == flag
    if mask.sum() < 10: continue
    kmf = KaplanMeierFitter()
    kmf.fit(journeys.loc[mask, 'duration_days'], journeys.loc[mask, 'purchased'], label=f'{label} (n={mask.sum():,})')
    kmf.plot_survival_function(ax=ax, color=color, linewidth=1.2, ci_show=True, ci_alpha=0.12)
    grp_d.append(journeys.loc[mask, 'duration_days']); grp_e.append(journeys.loc[mask, 'purchased'])
p_str = f"{logrank_test(grp_d[0], grp_d[1], grp_e[0], grp_e[1]).p_value:.4f}" if len(grp_d) == 2 else 'N/A'
ax.set_title('By Prior Purchase History', fontsize=11, fontweight='bold')
ax.set_xlabel(f'Days  (log-rank p={p_str})'); ax.set_ylabel('S(t)'); ax.set_xlim(left=0); ax.legend(fontsize=8)

# R2C2: Time of Day
ax = axes[1, 1]; add_overall(ax)
journeys['tod_bin'] = pd.cut(journeys['hour_of_day'], bins=[-1, 5, 11, 17, 23],
    labels=['Night (0-5)', 'Morning (6-11)', 'Afternoon (12-17)', 'Evening (18-23)'])
tod_colors = {'Night (0-5)': '#3949AB', 'Morning (6-11)': '#FFB300',
              'Afternoon (12-17)': '#43A047', 'Evening (18-23)': '#E53935'}
for label, color in tod_colors.items():
    mask = journeys['tod_bin'] == label
    if mask.sum() < 10: continue
    kmf = KaplanMeierFitter()
    kmf.fit(journeys.loc[mask, 'duration_days'], journeys.loc[mask, 'purchased'], label=f'{label} (n={mask.sum():,})')
    kmf.plot_survival_function(ax=ax, color=color, linewidth=1.2, ci_show=False)
ax.set_title('By Time of Day at First View', fontsize=11, fontweight='bold')
ax.set_xlabel('Days since first view'); ax.set_ylabel('S(t)'); ax.set_xlim(left=0); ax.legend(fontsize=8)

# R2C3: Price Tier
ax = axes[1, 2]; add_overall(ax)
p33, p67 = journeys['item_price'].quantile([0.33, 0.67])
journeys['price_tier'] = journeys['item_price'].apply(
    lambda p: f'Entry (<={p33:.0f})' if p <= p33 else (f'Mid ({p33:.0f}-{p67:.0f})' if p <= p67 else f'Premium (>{p67:.0f})')
)
tier_order  = [f'Entry (<={p33:.0f})', f'Mid ({p33:.0f}-{p67:.0f})', f'Premium (>{p67:.0f})']
tier_colors = ['#43A047', '#FFB300', '#E53935']
for tier, color in zip(tier_order, tier_colors):
    mask = journeys['price_tier'] == tier
    if mask.sum() < 10: continue
    kmf = KaplanMeierFitter()
    kmf.fit(journeys.loc[mask, 'duration_days'], journeys.loc[mask, 'purchased'], label=f'{tier} (n={mask.sum():,})')
    kmf.plot_survival_function(ax=ax, color=color, linewidth=1.2, ci_show=False)
ax.set_title('By Price Tier (tertiles)', fontsize=11, fontweight='bold')
ax.set_xlabel('Days since first view'); ax.set_ylabel('S(t)'); ax.set_xlim(left=0); ax.legend(fontsize=8)

# R2C4: Engagement Tier
ax = axes[1, 3]; add_overall(ax)
e33, e67 = journeys['log_engage_time'].quantile([0.33, 0.67])
journeys['eng_tier'] = journeys['log_engage_time'].apply(
    lambda v: 'Low' if v <= e33 else ('Mid' if v <= e67 else 'High')
)
for tier, color in zip(['Low', 'Mid', 'High'], ['#546E7A', '#FFB300', '#FF6F00']):
    mask = journeys['eng_tier'] == tier
    if mask.sum() < 10: continue
    kmf = KaplanMeierFitter()
    kmf.fit(journeys.loc[mask, 'duration_days'], journeys.loc[mask, 'purchased'], label=f'{tier} engagement (n={mask.sum():,})')
    kmf.plot_survival_function(ax=ax, color=color, linewidth=1.2, ci_show=False)
ax.set_title('By Engagement Time (tertiles)', fontsize=11, fontweight='bold')
ax.set_xlabel('Days since first view'); ax.set_ylabel('S(t)'); ax.set_xlim(left=0); ax.legend(fontsize=8)

fig.suptitle('Kaplan-Meier Survival Curves by Covariate', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# Log-rank test significance for each covariate grouping
tests = {
    'Item segment':  (journeys, 'segment'),
    'Time of day':   (journeys, 'tod_bin'),
    'Price tier':    (journeys, 'price_tier'),
    'Engagement':    (journeys, 'eng_tier'),
}
rows = []
for label, (jdf, col) in tests.items():
    data = jdf[jdf[col].notna()].copy()
    r = multivariate_logrank_test(data['duration_days'], data[col], data['purchased'])
    sig = '***' if r.p_value < 0.001 else '**' if r.p_value < 0.01 else '*' if r.p_value < 0.05 else 'ns'
    rows.append({'Test': label, 'N': len(data), 'df': r.degrees_of_freedom,
                 'chi2': round(r.test_statistic, 2), 'p-value': round(r.p_value, 4), 'Sig': sig})
display(pd.DataFrame(rows))

In [ ]:
# KM conversion milestones
sf = kmf_all.survival_function_
def pct_by(days):
    idx = sf.index.searchsorted(days)
    return (1 - sf.iloc[min(idx, len(sf)-1)]['KM_estimate']) * 100

milestones = pd.DataFrame({
    'Milestone': ['Within 1 day', 'Within 3 days', 'Within 7 days', 'Within 30 days'],
    '% Purchased': [f'{pct_by(d):.1f}%' for d in [1, 3, 7, 30]]
})
display(milestones)
print(f'Median survival time: {kmf_all.median_survival_time_}  (> 50% never purchase in window)')

### KM Key Takeaways

- **Buy fast or not at all:** 
  - ~40-50% of purchases happen within 24 hours of first view;
  the curve decelerates sharply after day 3. 
  - Median survival = infinity (fewer than 50%
  of viewed items are ever purchased).
- **Login status is the strongest splitter:** logged-in users convert at dramatically higher
  rates, reflecting purchase history (loyalty), not just authentication.
- **Mobile converts faster than desktop** despite smaller screen. 
  
  Mobile-first UX is the
  right priority.
- **High engagement accelerates conversion:** users who spend more time on the product
  page at first view convert faster and more.
- **Paid campaign traffic is slower:** sale, newsletter, retargeting channels all convert
  below organic hence suggest paid might just bring browsers, not buyers.

## 3.2 Cox Proportional Hazards Model

**Model:** $h(t) = h_0(t) \exp(\beta_1 x_1 + ... + \beta_p x_p)$

- $h_0(t)$: baseline hazard (how purchase risk evolves over time for an average user)
- $\exp(\beta X)$: covariates shift the baseline multiplicatively
- **HR > 1**: factor speeds up conversion; **HR < 1**: factor slows it down

**Two models:**
- **Step 3a (Baseline):** behavioral + contextual features only
- **Step 3b (Enriched):** same features + `session_item_sim` (NLP)

Comparing concordance indices shows the incremental value of session focus information.

### 3.2.1 Baseline model

In [ ]:
# Baseline Cox feature matrix (no NLP)
cox_cols_base = (['duration_days', 'purchased', 'log_price', 'is_mobile', 'is_logged_in',
                  'log_ga_session_num', 'has_prior_purchase', 'hour_of_day',
                  'log_engage_time'] + camp_cols + seg_cols)
cox_df_base = journeys[cox_cols_base].dropna().copy()

n_events = int(cox_df_base['purchased'].sum())
n_vars   = len(cox_cols_base) - 2
epv      = n_events / n_vars

epv_df = pd.DataFrame({
    'Metric': ['Events (purchases)', 'Covariates', 'EPV (events per variable)'],
    'Value':  [n_events, n_vars, f'{epv:.1f}']
})
display(epv_df)

In [ ]:
def highlight_pval(val):
    if val < 0.005: return 'background-color: #2d6a2d; color: white; font-weight: bold'
    elif val < 0.05: return 'background-color: #6aaa6a; color: white; font-weight: bold'
    else: return 'background-color: #8b0000; color: white'

# Initial baseline fit (pre-assumption check)
cph_pre = CoxPHFitter(penalizer=0.1)
cph_pre.fit(cox_df_base, duration_col='duration_days', event_col='purchased')
display(cph_pre.summary.style.applymap(highlight_pval, subset=['p']).format(precision=3)
        .set_caption('Initial Baseline Cox -- dark green p<0.005 | light green p<0.05 | red p>=0.05'))

In [ ]:
# Proportional hazards check (Schoenfeld residuals)
cph_pre.check_assumptions(cox_df_base, p_value_threshold=0.05, show_plots=False)

`has_prior_purchase` fails the Schoenfeld test: first-time vs repeat buyers have
genuinely different hazard shapes, not just different rates. 

**Stratification** gives
each group its own baseline hazard while all other covariates still produce valid HR estimates.

`hour_of_day` is not significant (p=0.55) and is dropped.

In [ ]:
# Stratified Baseline Cox (fixes PH violation)
strata_cols     = ['has_prior_purchase', 'is_logged_in']
drop_cols       = ['hour_of_day']
cox_cols_base_s = [c for c in cox_cols_base if c not in strata_cols + drop_cols]
cox_df_base_s   = journeys[cox_cols_base_s + strata_cols].dropna().copy()

cph_base = CoxPHFitter(penalizer=0.1)
cph_base.fit(cox_df_base_s, duration_col='duration_days', event_col='purchased', strata=strata_cols)

display(cph_base.summary.style.applymap(highlight_pval, subset=['p']).format(precision=3)
        .set_caption('Stratified Baseline Cox (has_prior_purchase stratified, hour_of_day dropped)'))
print(f'Baseline concordance index: {cph_base.concordance_index_:.4f}')

In [ ]:
# Baseline hazard ratio plot
fig, ax = plt.subplots(figsize=(9, 6))
summary = cph_base.summary.copy().sort_values('exp(coef)')
colors  = ['#43A047' if (r['p'] < 0.05 and r['exp(coef)'] > 1) else
           '#E53935' if (r['p'] < 0.05 and r['exp(coef)'] <= 1) else '#BDBDBD'
           for _, r in summary.iterrows()]
y_pos = range(len(summary))
ax.barh(y_pos, summary['exp(coef)'] - 1, left=1, color=colors, edgecolor='white', linewidth=0.8)
for i, (idx, row) in enumerate(summary.iterrows()):
    ax.plot([row['exp(coef) lower 95%'], row['exp(coef) upper 95%']], [i, i], color='#555', linewidth=1.5, zorder=5)
    ax.scatter(row['exp(coef)'], i, color='white', s=30, zorder=6, edgecolors='#333', linewidths=0.8)
ax.axvline(1, color='black', linewidth=1.2, linestyle='--', alpha=0.7)
ax.set_yticks(y_pos); ax.set_yticklabels(summary.index, fontsize=10)
ax.set_xlabel('Hazard Ratio  (HR > 1 = faster conversion)', fontsize=11)
ax.set_title('Baseline Cox PH -- Hazard Ratios with 95% CI (no NLP)', fontsize=13, fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='#43A047', label='Sig: speeds up (HR > 1)'),
    mpatches.Patch(color='#E53935', label='Sig: slows down (HR < 1)'),
    mpatches.Patch(color='#BDBDBD', label='Not significant'),
], fontsize=9, loc='lower right')
fig.tight_layout(); plt.show()

### 3.2.2 Enriched Cox Model (+ session_item_sim)

Same stratification and variable selection as baseline; `session_item_sim` added as one
additional covariate. Concordance comparison quantifies NLP lift.

In [ ]:
cox_cols_nlp = (['duration_days', 'purchased', 'log_price', 'is_mobile',
                 'log_ga_session_num', 'has_prior_purchase', 'hour_of_day',
                 'log_engage_time', 'session_item_sim'] + camp_cols + seg_cols)
cox_cols_s = [c for c in cox_cols_nlp if c not in strata_cols + drop_cols]
cox_df_s   = journeys[cox_cols_s + strata_cols].dropna().copy()

n_events_s = int(cox_df_s['purchased'].sum())
n_vars_s   = len(cox_cols_s) - 2

cph = CoxPHFitter(penalizer=0.1)
cph.fit(cox_df_s, duration_col='duration_days', event_col='purchased', strata=strata_cols)

display(cph.summary.style.applymap(highlight_pval, subset=['p']).format(precision=3)
        .set_caption('Enriched Cox PH (+ session_item_sim)'))
print(f'Enriched concordance index: {cph.concordance_index_:.4f}')

In [ ]:
def group_lrt(full_model, df, drop_group, duration_col, event_col, strata):
    """Likelihood-ratio test: full Cox vs reduced model with one group of dummies removed."""
    keep = [c for c in df.columns if c not in drop_group]
    red = CoxPHFitter(penalizer=0.1)
    red.fit(df[keep], duration_col=duration_col, event_col=event_col, strata=strata)
    stat = 2 * (full_model.log_likelihood_ - red.log_likelihood_)
    p    = chi2.sf(stat, df=len(drop_group))
    return stat, len(drop_group), p

stat_c, df_c, p_c = group_lrt(cph, cox_df_s, camp_cols, "duration_days", "purchased", strata_cols)
stat_s, df_s, p_s = group_lrt(cph, cox_df_s, seg_cols,  "duration_days", "purchased", strata_cols)

lrt_df = pd.DataFrame({
    "Group":     ["Campaign type", "Item segment"],
    "Dummies":   [df_c,  df_s],
    "chi2":      [round(stat_c, 2), round(stat_s, 2)],
    "p (joint)": [p_c,  p_s],
})
display(lrt_df.style
    .format({"chi2": "{:.2f}", "p (joint)": "{:.4f}"})
    .applymap(lambda v: "background-color:#2d6a2d;color:white" if v < 0.05
              else "background-color:#8b0000;color:white", subset=["p (joint)"])
    .set_caption("Joint LRT: is the group as a whole significant?"))


In [ ]:
# Baseline vs Enriched comparison
sim_row = cph.summary.loc['session_item_sim']
comp_cox = pd.DataFrame({
    'Model':              ['Baseline Cox (no NLP)', 'Enriched Cox (+ session_item_sim)'],
    'Concordance Index':  [round(cph_base.concordance_index_, 4), round(cph.concordance_index_, 4)],
    'Lift vs Baseline':   [0.0, round(cph.concordance_index_ - cph_base.concordance_index_, 4)],
})
display(comp_cox.style.format({'Concordance Index': '{:.4f}', 'Lift vs Baseline': '{:+.4f}'}))
print(f'session_item_sim: HR={sim_row["exp(coef)"]:.2f}  '
      f'(95% CI {sim_row["exp(coef) lower 95%"]:.2f}-{sim_row["exp(coef) upper 95%"]:.2f}, '
      f'p={sim_row["p"]:.4f})')

In [ ]:
# Enriched hazard ratio plot
fig, ax = plt.subplots(figsize=(9, 6))
summary = cph.summary.copy().sort_values('exp(coef)')
colors  = ['#43A047' if (r['p'] < 0.05 and r['exp(coef)'] > 1) else
           '#E53935' if (r['p'] < 0.05 and r['exp(coef)'] <= 1) else '#BDBDBD'
           for _, r in summary.iterrows()]
y_pos = range(len(summary))
ax.barh(y_pos, summary['exp(coef)'] - 1, left=1, color=colors, edgecolor='white', linewidth=0.8)
for i, (idx, row) in enumerate(summary.iterrows()):
    ax.plot([row['exp(coef) lower 95%'], row['exp(coef) upper 95%']], [i, i], color='#555', linewidth=1.5, zorder=5)
    ax.scatter(row['exp(coef)'], i, color='white', s=30, zorder=6, edgecolors='#333', linewidths=0.8)
ax.axvline(1, color='black', linewidth=1.2, linestyle='--', alpha=0.7)
ax.set_yticks(y_pos); ax.set_yticklabels(summary.index, fontsize=10)
ax.set_xlabel('Hazard Ratio  (HR > 1 = faster conversion)', fontsize=11)
ax.set_title('Enriched Cox PH -- Hazard Ratios with 95% CI (+ session_item_sim)', fontsize=13, fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='#43A047', label='Sig: speeds up (HR > 1)'),
    mpatches.Patch(color='#E53935', label='Sig: slows down (HR < 1)'),
    mpatches.Patch(color='#BDBDBD', label='Not significant'),
], fontsize=9, loc='lower right')
fig.tight_layout(); plt.show()

### 3.2.3 Survival Curves by Factor

Each subplot holds all other covariates at baseline and varies one factor.

**Baseline:** first-time buyer, mobile, EUR 180 item, session 1, organic, accessories, not logged in.

In [ ]:
BASE = {
    'log_price':          np.log1p(180),
    'is_mobile':          1,
    'log_ga_session_num': np.log1p(1),
    'log_engage_time':    np.log1p(5000),
    'session_item_sim':   journeys['session_item_sim'].median(),
    'has_prior_purchase': 0,
    'is_logged_in':      0,
}
for c in seg_cols + camp_cols:
    BASE[c] = 0

def sf_row(**kwargs):
    p = BASE.copy(); p.update(kwargs)
    return cph.predict_survival_function(pd.DataFrame([p]))

XLIM = 45
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Price tier
ax = axes[0, 0]
for price, color in zip([100, 300, 500, 800, 1500], ['#43A047','#8BC34A','#FFB300','#FF5722','#B71C1C']):
    sf = sf_row(log_price=np.log1p(price))
    ax.plot(sf.index, sf.values.flatten(), color=color, linewidth=2, label=f'EUR {price}')
ax.set_title('Price', fontsize=11, fontweight='bold'); ax.set_xlim(0, XLIM)
ax.set_xlabel('Days'); ax.set_ylabel('P(not yet purchased)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Campaign channel
ax = axes[0, 1]
channel_specs = [
    ('Organic',       {}, '#1E88E5'),
    ('Sale',          {'camp_sale': 1}, '#E53935'),
    ('Newsletter',    {'camp_newsletter': 1}, '#43A047'),
    ('Retargeting',   {'camp_retargeting': 1}, '#9C27B0'),
    ('Paid search',   {'camp_paid_search': 1}, '#FF9800'),
]
for label, spec, color in channel_specs:
    sf = sf_row(**spec)
    ax.plot(sf.index, sf.values.flatten(), color=color, linewidth=2, label=label)
ax.set_title('Acquisition Channel', fontsize=11, fontweight='bold'); ax.set_xlim(0, XLIM)
ax.set_xlabel('Days'); ax.set_ylabel('P(not yet purchased)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Device
ax = axes[0, 2]
for label, spec, color in [('Mobile', {'is_mobile': 1}, '#1E88E5'), ('Desktop', {'is_mobile': 0}, 'goldenrod')]:
    sf = sf_row(**spec)
    ax.plot(sf.index, sf.values.flatten(), color=color, linewidth=2, label=label)
ax.set_title('Device', fontsize=11, fontweight='bold'); ax.set_xlim(0, XLIM)
ax.set_xlabel('Days'); ax.set_ylabel('P(not yet purchased)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Session number (visit #)
ax = axes[1, 0]
for sess, color in zip([1, 2, 4, 8, 15], ['#1E88E5','#43A047','#FFB300','#FF5722','#B71C1C']):
    sf = sf_row(log_ga_session_num=np.log1p(sess))
    ax.plot(sf.index, sf.values.flatten(), color=color, linewidth=2, label=f'Session {sess}')
ax.set_title('Visit Number', fontsize=11, fontweight='bold'); ax.set_xlim(0, XLIM)
ax.set_xlabel('Days'); ax.set_ylabel('P(not yet purchased)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Engagement time
ax = axes[1, 1]
for ms, label, color in [(500,'Low',  '#546E7A'), (3000,'Med','#FFB300'),
                          (10000,'High','#FF6F00'), (30000,'Very High','#B71C1C')]:
    sf = sf_row(log_engage_time=np.log1p(ms))
    ax.plot(sf.index, sf.values.flatten(), color=color, linewidth=2, label=label)
ax.set_title('Engagement Time', fontsize=11, fontweight='bold'); ax.set_xlim(0, XLIM)
ax.set_xlabel('Days'); ax.set_ylabel('P(not yet purchased)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Purchase history
ax = axes[1, 2]
for label, hist, color in [('First-time buyer', 0, '#1E88E5'), ('Repeat buyer', 1, '#E53935')]:
    sf = sf_row(has_prior_purchase=hist)
    ax.plot(sf.index, sf.values.flatten(), color=color, linewidth=2, label=label)
ax.set_title('Purchase History', fontsize=11, fontweight='bold'); ax.set_xlim(0, XLIM)
ax.set_xlabel('Days'); ax.set_ylabel('P(not yet purchased)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

fig.suptitle('Predicted Survival Curves -- Effect of Each Covariate (Enriched Cox)',
             fontsize=13, fontweight='bold')
fig.tight_layout(); plt.show()

### 3.2.4 Buyer Archetypes

Each profile maps to one of the four established buyer archetypes
(Schottmuller, Unbounce CTA Conference) using Cox model covariates
and dataset-calibrated price points (median purchase: EUR 180).

| Archetype       | Device  | Channel        | Visit # | Engagement | Price   | History      |
|-----------------|---------|----------------|---------|------------|---------|--------------|
| **Spontaneous** | Mobile  | Sale campaign  | 1st     | High       | EUR 120 | First-time   |
| **Humanistic**  | Mobile  | Organic/Retarg | 2nd–3rd | Medium     | EUR 250 | Repeat buyer |
| **Methodical**  | Desktop | Organic        | 7th+    | Medium     | EUR 500 | First-time   |
| **Competitive** | Desktop | Organic        | 5th+    | High       | EUR 800 | First-time   |

**Archetype Validation: Pairwise Log-Rank Tests**

Archetypes are classified on observed journeys and log-rank tests check whether
each pair represents statistically distinct survival curves.

In [ ]:
import itertools
 
 
def classify_archetype(row):
    sessions = np.expm1(row['log_ga_session_num'])
    price    = np.expm1(row['log_price'])
 
    # 1. Spontaneous: sale-triggered mobile purchase of an entry-level item
    #    on the very first or second visit — genuinely impulsive behaviour.
    if (
        row.get('camp_sale', 0) == 1
        and row['is_mobile'] == 1
        and price <= 200
        and sessions <= 2
    ):
        return 'Spontaneous'
 
    # 2. Humanistic: returning customer (brand relationship) OR retargeted
    #    mobile user at a mid-range price (familiarity drives re-engagement).
    if row['has_prior_purchase'] == 1 or (
        row.get('camp_retargeting', 0) == 1
        and row['is_mobile'] == 1
        and price <= 400
    ):
        return 'Humanistic'
 
    # 3. Competitive: desktop buyer targeting higher-end items (≥ EUR 600),
    #    has done some research but is decisive — status and quality driven.
    if (
        row['is_mobile'] == 0
        and price >= 600
        and sessions >= 3
        and row.get('camp_sale', 0) == 0
    ):
        return 'Competitive'
 
    # 4. Methodical: desktop buyer in the mid-price band (EUR 200–600),
    #    many sessions, no campaign — systematic researcher.
    if (
        row['is_mobile'] == 0
        and sessions >= 3
        and 200 <= price < 600
        and row.get('camp_sale', 0) == 0
        and row.get('camp_retargeting', 0) == 0
    ):
        return 'Methodical'
 
    return None
 
 
arc_df = cox_df_s.copy()
arc_df['archetype'] = arc_df.apply(classify_archetype, axis=1)
arc_df = arc_df[arc_df['archetype'].notna()]
 
print('Archetype group sizes:')
display(
    arc_df.groupby('archetype')[['duration_days', 'purchased']]
    .agg(n=('duration_days', 'count'), events=('purchased', 'sum'))
    .reset_index()
)
 
# ── pairwise log-rank p-value matrix ─────────────────────────────────────────
archetypes = sorted(arc_df['archetype'].unique().tolist())
n    = len(archetypes)
pmat = np.ones((n, n))
 
for i, a1 in enumerate(archetypes):
    for j, a2 in enumerate(archetypes):
        if i >= j:
            continue
        g1 = arc_df[arc_df['archetype'] == a1]
        g2 = arc_df[arc_df['archetype'] == a2]
        r  = logrank_test(
            g1['duration_days'], g2['duration_days'],
            g1['purchased'],     g2['purchased'],
        )
        pmat[i, j] = pmat[j, i] = r.p_value
 
 
def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
 
 
# ── figure: pairwise log-rank heatmap ────────────────────────────────────────
log_pmat = -np.log10(np.where(np.eye(n, dtype=bool), np.nan, pmat))
fig, ax  = plt.subplots(figsize=(8, 6))
im = ax.imshow(log_pmat, cmap='RdYlGn', vmin=0, vmax=4, aspect='auto')
ax.set_xticks(range(n))
ax.set_xticklabels(archetypes, fontsize=9, rotation=20, ha='right')
ax.set_yticks(range(n))
ax.set_yticklabels(archetypes, fontsize=9)
 
for i in range(n):
    for j in range(n):
        if i == j:
            ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, color='#cccccc'))
            ax.text(j, i, '--', ha='center', va='center', fontsize=12, color='#888')
        else:
            p = pmat[i, j]
            ax.text(
                j, i, f'p={p:.3f}\n{stars(p)}',
                ha='center', va='center', fontsize=9, fontweight='bold',
                color='white' if log_pmat[i, j] > 2 else 'black',
            )
 
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('-log10(p)', fontsize=10)
cbar.set_ticks([0, 1, 2, 3, 4])
cbar.set_ticklabels(['p=1', 'p=0.1', 'p=0.01', 'p=0.001', 'p=0.0001'])
ax.set_title('Pairwise Log-Rank Tests Between Archetypes', fontsize=12, fontweight='bold')
ax.grid(False)
fig.tight_layout()
plt.show()
 
pairs = list(itertools.combinations(range(n), 2))
print(
    f'{sum(pmat[i, j] < 0.05 for i, j in pairs)}/{len(pairs)} '
    f'pairs are statistically distinct (p < 0.05)'
)

In [ ]:
PROFILES = {
    # Spontaneous: triggered by a sale campaign on mobile during the first visit.
    # Targets entry-level items (sunglasses, small accessories ~EUR 120).
    # Very high engagement on the product page signals immediate intent.
    'Spontaneous': {
        'log_price':          np.log1p(120),
        'is_mobile':          1,
        'log_ga_session_num': np.log1p(1),
        'log_engage_time':    np.log1p(40000),   # deeply absorbed on first view
        'has_prior_purchase': 0,
        'camp_sale':          1,
    },
 
    # Humanistic: relationship-driven buyer — either a returning customer or
    # someone re-engaged via retargeting. Buys mid-range items (~EUR 250,
    # e.g. wallet, belt, small crossbody) driven by brand familiarity.
    'Humanistic': {
        'log_price':          np.log1p(250),
        'is_mobile':          1,
        'log_ga_session_num': np.log1p(2),
        'log_engage_time':    np.log1p(6000),
        'has_prior_purchase': 1,                 # prior relationship is the key signal
        'camp_retargeting':   1,
    },
 
    # Methodical: desktop researcher who systematically evaluates options before
    # buying. Mid-range item (~EUR 500, e.g. shoes, small bag). Many sessions,
    # moderate engagement per visit — careful, not impulsive.
    'Methodical': {
        'log_price':          np.log1p(500),
        'is_mobile':          0,
        'log_ga_session_num': np.log1p(7),       # many return visits before deciding
        'log_engage_time':    np.log1p(3000),    # steady but not obsessive
        'has_prior_purchase': 0,
    },
 
    # Competitive: status-driven desktop buyer targeting higher-end items
    # (~EUR 800, e.g. La Medusa bag, premium shoes). Fewer sessions than
    # Methodical — they know what they want — but very high engagement because
    # they scrutinise quality and prestige closely.
    'Competitive': {
        'log_price':          np.log1p(800),
        'is_mobile':          0,
        'log_ga_session_num': np.log1p(5),
        'log_engage_time':    np.log1p(10000),   # high: inspecting a premium item
        'has_prior_purchase': 0,
    },
}

PROFILE_COLORS = {
    'Spontaneous': '#E53935',   # red    — urgency, energy
    'Humanistic':  '#FF9800',   # amber  — warmth, relationship
    'Methodical':  '#43A047',   # green  — steady, logical
    'Competitive': '#1565C0',   # blue   — premium, status
}

def build_row(spec):
    row = BASE.copy(); row.update(spec)
    return pd.DataFrame([row])

sfs = {name: cph.predict_survival_function(build_row(spec)) for name, spec in PROFILES.items()}

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

ax = axes[0]
for name, sf in sfs.items():
    ax.plot(sf.index, sf.values.flatten(), color=PROFILE_COLORS[name], linewidth=2.5, label=name)
ax.set_title('Buyer Archetypes: Survival Curves', fontsize=12, fontweight='bold')
ax.set_xlabel('Days since first view'); ax.set_ylabel('P(not yet purchased)')
ax.set_xlim(0, 45); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)

DAYS = [1, 3, 7, 14, 30]
def pct_by(sf, day):
    idx = sf.index.searchsorted(day)
    return round((1 - sf.iloc[min(idx, len(sf)-1)].values[0]) * 100, 1)
heat_data = pd.DataFrame(
    {name: [pct_by(sfs[name], d) for d in DAYS] for name in PROFILES},
    index=[f'Day {d}' for d in DAYS]
).T
ax = axes[1]
im = ax.imshow(heat_data.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(DAYS))); ax.set_xticklabels([f'Day {d}' for d in DAYS], fontsize=10)
ax.set_yticks(range(len(PROFILES))); ax.set_yticklabels(list(PROFILES.keys()), fontsize=10)
for i in range(len(PROFILES)):
    for j in range(len(DAYS)):
        val = heat_data.values[i, j]
        ax.text(j, i, f'{val:.1f}%', ha='center', va='center', fontsize=11, fontweight='bold',
                color='white' if val > heat_data.values.max() * 0.6 else 'black')
plt.colorbar(im, ax=ax, label='% converted')
ax.set_title('% Converted by Milestone Day', fontsize=12, fontweight='bold')
ax.grid(False)
fig.suptitle('Buyer Archetypes: Conversion Speed', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

Archetypes|Profile| Implication|
---|---|---|
Spontaneous | first visit, mobile, sale-triggered, low price| converts fastest and will not return if missed|
Humanistic | recent buyer or retargeted mobile user, mid-range price| converts on familiarity and brand relationship|
Methodical | desktop, many sessions, no campaign, mid-range price| longest consideration cycle, needs depth|
Competitive |desktop, high engagement, high price| decisive but scrutinises carefully before committing|

## 3.3 ML Survival Models: XGBoost AFT & Random Survival Forest

Two tree-based survival models extend the Cox framework by relaxing linearity.

| Model | Core assumption | Output |
|-------|-----------------|--------|
| **XGBoost AFT** | Parametric (Normal); trees model `log(time)` directly | Expected time-to-purchase |
| **Random Survival Forest** | None — ensemble of survival trees | Survival function per user |

Evaluated on **concordance index (C-index)**: fraction of user pairs ranked correctly by conversion speed.  
*(0.5 = random guessing, 1.0 = perfect)*

In [ ]:
# Feature set: enriched (same as s3.3 Cox + NLP), strata vars kept as regular features
feat_cols_ml = [c for c in cox_cols_nlp if c not in ['duration_days', 'purchased']]
surv_ml = journeys[cox_cols_nlp].dropna().copy()

X_all = surv_ml[feat_cols_ml].values.astype(float)
t_all = surv_ml['duration_days'].values.astype(float)
e_all = surv_ml['purchased'].values.astype(bool)

X_tr, X_te, t_tr, t_te, e_tr, e_te = train_test_split(
    X_all, t_all, e_all, test_size=0.2, random_state=42
)
print(f'Train: {len(X_tr):,}  |  Test: {len(X_te):,}')
print(f'Test purchases: {e_te.sum():,} / {len(e_te):,} ({e_te.mean():.1%})')

In [ ]:
# XGBoost AFT
def _aft_dm(X, t, e):
    dm = xgb.DMatrix(X, feature_names=feat_cols_ml)
    dm.set_float_info('label_lower_bound', t)
    dm.set_float_info('label_upper_bound', np.where(e, t, np.inf))
    return dm

dtrain_aft = _aft_dm(X_tr, t_tr, e_tr)
dtest_aft  = _aft_dm(X_te, t_te, e_te)

xgb_aft_model = xgb.train(
    {'objective': 'survival:aft', 'eval_metric': 'aft-nloglik',
     'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 1.0,
     'max_depth': 4, 'learning_rate': 0.05,
     'subsample': 0.8, 'colsample_bytree': 0.8, 'seed': 42},
    dtrain_aft,
    num_boost_round=300,
    evals=[(dtest_aft, 'test')],
    early_stopping_rounds=20,
    verbose_eval=50
)

t_pred_aft = xgb_aft_model.predict(dtest_aft)
ci_aft = concordance_index(t_te, t_pred_aft, e_te)
print(f'\nXGBoost AFT  C-index: {ci_aft:.4f}')

In [ ]:
# Random Survival Forest
if HAS_SKSURV:
    y_tr_surv = Surv.from_arrays(event=e_tr, time=t_tr)
    y_te_surv = Surv.from_arrays(event=e_te, time=t_te)

    rsf = RandomSurvivalForest(
        n_estimators=200, max_depth=8, min_samples_split=20,
        n_jobs=-1, random_state=42
    )
    rsf.fit(X_tr, y_tr_surv)
    ci_rsf = rsf.score(X_te, y_te_surv)
    print(f'RSF  C-index: {ci_rsf:.4f}')
else:
    ci_rsf = None
    print('RSF skipped')

In [ ]:
# ── Model comparison table ────────────────────────────────────────────────────
rows = [
    ('Cox PH (stratified)', 'Semi-parametric (linear)', cph_base.concordance_index_),
    ('Cox PH + NLP',        'Semi-parametric (linear)', cph.concordance_index_),
    ('XGBoost AFT',         'ML - parametric',          ci_aft),
]
if ci_rsf is not None:
    rows.append(('Random Survival Forest', 'ML - non-parametric', ci_rsf))

comp_df = (pd.DataFrame(rows, columns=['Model', 'Type', 'C-index'])
             .sort_values('C-index', ascending=False)
             .reset_index(drop=True))
display(comp_df.style
        .format({'C-index': '{:.4f}'})
        .bar(subset='C-index', color='#4472c4', vmin=0.5, vmax=1.0)
        .set_caption('Survival Model Comparison - Concordance Index (test set)'))

import shap
from sklearn.inspection import permutation_importance

# manual permutation importance helper (works with any predict fn + C-index)
def _perm_imp(predict_fn, X, t, e, n_repeats=5, seed=42):
    rng = np.random.RandomState(seed)
    base = concordance_index(t, predict_fn(X), e)
    imp = np.zeros((n_repeats, X.shape[1]))
    for r in range(n_repeats):
        for j in range(X.shape[1]):
            Xp = X.copy(); Xp[:, j] = rng.permutation(Xp[:, j])
            imp[r, j] = base - concordance_index(t, predict_fn(Xp), e)
    return imp.mean(axis=0)

aft_predict = lambda X: xgb_aft_model.predict(xgb.DMatrix(X, feature_names=feat_cols_ml))


# ── XGBoost AFT: SHAP ─────────────────────────────────────────────────────────
print("XGBoost AFT - SHAP (TreeExplainer)...")
explainer_aft = shap.TreeExplainer(xgb_aft_model)
shap_vals_aft = explainer_aft.shap_values(xgb.DMatrix(X_te, feature_names=feat_cols_ml))

fig, axs = plt.subplots(1, 2, figsize=(15, 6))

ax = axs[0]
plt.sca(ax)   # FIX: tell SHAP to plot on the left subplot

shap.summary_plot(
    shap_vals_aft,
    X_te,
    feature_names=feat_cols_ml,
    plot_type='dot',
    show=False,
    max_display=12,
    plot_size=None   # FIX: prevents SHAP from resizing/creating its own layout
)

ax.set_title('XGBoost AFT - SHAP Beeswarm (test set)', pad=20)

ax = axs[1]
# ── XGBoost AFT: Permutation importance ───────────────────────────────────────
print("XGBoost AFT - Permutation importance...")
aft_pi_vals = _perm_imp(aft_predict, X_te, t_te, e_te)
aft_pi = pd.Series(aft_pi_vals, index=feat_cols_ml).sort_values(ascending=False).head(12)

ax.barh(aft_pi.index[::-1], aft_pi.values[::-1], color='#4472c4')
ax.set_title('XGBoost AFT - Permutation Importance (C-index drop)')
ax.set_xlabel('Mean drop in C-index')

plt.tight_layout()
plt.show()

The C-index lift from Cox → ML is modest, confirming that the linear Cox model already
  captures most of the signal. 
  
  The primary value of ML here is **confirmation and feature
  importance robustness**, not raw predictive gain.

## 3.4 Key Findings

### Model performance

All four models were evaluated on a held-out test set using the concordance index (C-index — fraction of user pairs correctly ranked by conversion speed; 0.5 = random, 1.0 = perfect)

XGBoost AFT and RSF outperform both Cox variants, though the gap is modest. The linear Cox model already captures most of the signal

### Overall conversion pattern

The conversion window is narrow: most purchases happen within the first 3 days of initial item view. After that, the probability drops sharply and plateaus. 

Users who don't convert quickly are unlikely to convert at all within the same consideration cycle

### What drives conversion speed

- **Browsing focus (session_item_sim)**: users who concentrate on a narrow, coherent set of items convert faster — scattered browsing across unrelated categories is a strong signal of non-purchase

- **Item price**: higher-priced items take longer to convert, but the relationship is not simply linear — entry-level accessories convert quickly regardless of channel, while mid-to-high price points show more variation depending on device and visit history

- **Prior purchase history** and **session number** consistently matter: returning customers and users deeper into their consideration journey convert faster

# 4. Binary Session Classification

**Input data:** `session_df` was built in above — one row per (user × session)

**Question:** Can we predict whether a session results in a purchase,
using in-session behaviour and user history?

**Context:** All users in the TTP dataset are eventual purchasers. The model predicts
which sessions convert -- both first-time purchases and repeat purchases. `has_purchased_before`
lets the model learn different patterns for each group, analogous to how the Cox model
stratifies on `has_prior_purchase`.

**Models compared:** Logistic Regression *(linear baseline)*, Random Forest, XGBoost, HistGBM, Soft-Vote Ensemble.

Rationale: Purchase behavior is non-linear and interaction-driven, favoring tree models. HistGBM additionally handles structural missings (e.g. days_since_prev_session = NaN for first sessions) natively. Logistic Regression benchmarks whether non-linearity meaningfully improves performance.

## 4.1 Feature & Target Setup

**Target:** `purchase_in_session` — 1 if the session ended in a purchase, 0 if the user browsed and left.

UTM nulls are filled to `'direct'` before the pipeline so organic traffic is not treated as missing data.

In [ ]:
target = 'purchase_in_session'
feature_cols = [
    'device_category', 'login_status',
    'utm_flag', 'utm_source', 'utm_campaign',
    'first_hour', 'weekday', 'ga_session_number', 'main_item_category',
    'n_events', 'n_unique_items', 'n_view_item', 'n_add_to_cart',
    'avg_item_price', 'max_item_price', 'avg_engagement_time_msec',
    'session_duration_sec', 'add_to_view_ratio',
    'prev_sessions', 'prev_purchases', 'has_purchased_before',
    'days_since_prev_session',
]
categorical_cols = ['device_category', 'login_status', 'utm_source', 'utm_campaign',
                    'weekday', 'main_item_category']
numeric_cols     = [c for c in feature_cols if c not in categorical_cols]

model_df = session_df[['user_pseudo_id'] + feature_cols + [target]].copy()
for col in categorical_cols:
    model_df[col] = model_df[col].astype('object').where(pd.notnull(model_df[col]), None)

model_df[categorical_cols] = model_df[categorical_cols].replace('(not set)', np.nan)

model_df[target] = pd.to_numeric(model_df[target], errors='coerce')
model_df = model_df.dropna(subset=[target]).copy()
model_df[target] = model_df[target].astype(int)

X = model_df[feature_cols].copy()
y = model_df[target].copy()
groups = model_df['user_pseudo_id'].copy()
print(f'Model dataset: {len(model_df):,} sessions from {groups.nunique():,} users \n {y.sum():,} positive cases')

## 4.2 Train / Test Split

Split is **group-aware** on `user_pseudo_id` via `GroupShuffleSplit`. All sessions from a given user land in the same partition, so the model never sees the same user in both train and test. A random split would leak cross-session patterns and overstate performance.

In [ ]:
# Group-aware split: all sessions from one user land in the same set
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

split_summary = pd.DataFrame({
    'Set':      ['Train', 'Test'],
    'Sessions': [len(X_train), len(X_test)],
    'Purchases': [y_train.sum(), y_test.sum()],
    'Purchase %': [f'{y_train.mean()*100:.2f}%', f'{y_test.mean()*100:.2f}%']
})
display(split_summary)

Split by `user_pseudo_id`: all sessions from a user land in one set.
This prevents leakage from cross-session patterns and gives a realistic out-of-sample estimate.

## 4.3 Model Training

### Baseline 

Four classifiers, each wrapped in a fresh preprocessing pipeline (median imputation for numerics, ordinal encoding for categoricals). Class imbalance is handled natively rather than with SMOTE:

- **LR & RF:** `class_weight='balanced'` re-weights the loss per class
- **XGBoost:** `scale_pos_weight` = negative / positive count
- **HistGBM:** `class_weight='balanced'`; also handles structural NaNs (e.g. `days_since_prev_session` is NaN for first sessions) without imputation

A **soft-vote ensemble** averages the predicted probabilities of all four models.

> Threshold selection is not performed here
> 
> The TTP sample consists entirely of eventual purchasers, making calibrated probability estimates unsuitable for deployment without revalidation on a representative population.

In [ ]:
# Shared preprocessing factory (fresh instance per pipeline to avoid state sharing)
def make_preprocessor():
    return ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), categorical_cols)
    ])

In [ ]:
# Class imbalance weight for XGBoost
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    'Logistic Regression': Pipeline([
        ('prep',  make_preprocessor()),
        ('scale', StandardScaler()),
        ('clf',   LogisticRegression(class_weight='balanced', max_iter=1000, C=0.1, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('prep', make_preprocessor()),
        ('clf',  RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced',
                                         min_samples_leaf=20, random_state=42, n_jobs=-1))
    ]),
    'XGBoost': Pipeline([
        ('prep', make_preprocessor()),
        ('clf',  xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                    scale_pos_weight=pos_weight, eval_metric='aucpr',
                                    tree_method='hist', random_state=42, verbosity=0))
    ]),
    'HistGBM': Pipeline([
        ('prep', make_preprocessor()),
        ('clf',  HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=6,
                                                 min_samples_leaf=20, class_weight='balanced',
                                                 random_state=42))
    ]),
}

In [ ]:
# Train all models and collect test-set scores
results = {}
for name, pipe in models.items():
    print(f'Fitting {name}...')
    pipe.fit(X_train, y_train)
    probs = pipe.predict_proba(X_test)[:, 1]
    results[name] = {
        'pipe': pipe, 'probs': probs,
        'roc': roc_auc_score(y_test, probs),
        'pr':  average_precision_score(y_test, probs)
    }
    print(f'  ROC AUC={results[name]["roc"]:.4f}  PR AUC={results[name]["pr"]:.4f}')

# Soft-vote ensemble
ens_probs = np.mean([r['probs'] for r in results.values()], axis=0)
results['Ensemble (soft vote)'] = {
    'probs': ens_probs,
    'roc':   roc_auc_score(y_test, ens_probs),
    'pr':    average_precision_score(y_test, ens_probs)
}

comp_df = pd.DataFrame([
    {'Model': name, 'ROC AUC': r['roc'], 'PR AUC': r['pr']}
    for name, r in results.items()
]).sort_values('PR AUC', ascending=False).reset_index(drop=True)

display(comp_df.style
    .format({'ROC AUC': '{:.4f}', 'PR AUC': '{:.4f}'})
    .background_gradient(subset=['ROC AUC', 'PR AUC'], cmap='Greens')
    .set_caption('Model Comparison -- Test Set Performance (sorted by PR AUC)'))

In [ ]:
# For interpretability, we'll take the best individual model (not ensemble) forward for SHAP analysis
indiv = comp_df[~comp_df['Model'].str.contains('Ensemble')]
best_name = indiv.iloc[0]['Model']
best_pipe = results[best_name]['pipe']

### Adding item similarity to the model

We now add `session_item_sim` — the average pairwise cosine similarity of sentence-transformer embeddings for all items viewed in the session — to the best-performing baseline classifier. High similarity signals focused, intent-driven browsing; we expect it to push purchasing sessions higher in the ranking.

The enriched model uses the **same train/test split** as the baseline so the PR-AUC comparison is apples-to-apples.

In [ ]:
from sklearn.base import clone

In [ ]:
# ── 1. Merge session_item_sim onto session_df ────────────────────────────────
# session_viewed and med_sim were computed in §1.4 (NLP Enrichment, Feature Engineering)
session_df_nlp = session_df.merge(
    session_viewed[['session_id', 'session_item_sim']],
    on='session_id', how='left'
)
session_df_nlp['session_item_sim'] = session_df_nlp['session_item_sim'].fillna(med_sim)

# ── 2. Rebuild feature matrix with session_item_sim added ────────────────────
feature_cols_nlp  = feature_cols + ['session_item_sim']
numeric_cols_nlp  = numeric_cols  + ['session_item_sim']

model_df_nlp = session_df_nlp[['user_pseudo_id'] + feature_cols_nlp + [target]].copy()
for col in categorical_cols:
    model_df_nlp[col] = model_df_nlp[col].astype('object').where(pd.notnull(model_df_nlp[col]), None)
model_df_nlp[categorical_cols] = model_df_nlp[categorical_cols].replace('(not set)', np.nan)
model_df_nlp[target] = pd.to_numeric(model_df_nlp[target], errors='coerce')
model_df_nlp = model_df_nlp.dropna(subset=[target]).copy()
model_df_nlp[target] = model_df_nlp[target].astype(int)

X_nlp    = model_df_nlp[feature_cols_nlp].copy()
y_nlp    = model_df_nlp[target].copy()
groups_nlp = model_df_nlp['user_pseudo_id'].copy()

# ── 3. Same group-aware split (re-derive to match NLP row order) ─────────────
splitter_nlp = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx_nlp, test_idx_nlp = next(splitter_nlp.split(X_nlp, y_nlp, groups=groups_nlp))
X_train_nlp, X_test_nlp = X_nlp.iloc[train_idx_nlp], X_nlp.iloc[test_idx_nlp]
y_train_nlp, y_test_nlp = y_nlp.iloc[train_idx_nlp], y_nlp.iloc[test_idx_nlp]

# ── 4. Retrain best baseline model with enriched features ────────────────────
def make_preprocessor_nlp():
    return ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric_cols_nlp),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), categorical_cols)
    ])

# Clone best model's classifier with same hyperparameters
best_clf = results[best_name]['pipe'].named_steps['clf']

steps_nlp = [('prep', make_preprocessor_nlp())]
if 'scale' in results[best_name]['pipe'].named_steps:
    steps_nlp.append(('scale', StandardScaler()))
steps_nlp.append(('clf', clone(best_clf)))

pipe_nlp = Pipeline(steps_nlp)
pipe_nlp.fit(X_train_nlp, y_train_nlp)
probs_nlp = pipe_nlp.predict_proba(X_test_nlp)[:, 1]

pr_nlp  = average_precision_score(y_test_nlp, probs_nlp)
roc_nlp = roc_auc_score(y_test_nlp, probs_nlp)

pr_base  = results[best_name]['pr']
roc_base = results[best_name]['roc']

# ── 5. Comparison table ───────────────────────────────────────────────────────
comp_nlp = pd.DataFrame({
    'Model':     [f'{best_name} (baseline)', f'{best_name} + session_item_sim'],
    'PR-AUC':    [pr_base,  pr_nlp],
    'ROC-AUC':   [roc_base, roc_nlp],
    'PR-AUC lift': [0.0, pr_nlp - pr_base],
})
display(comp_nlp.style
    .format({'PR-AUC': '{:.4f}', 'ROC-AUC': '{:.4f}', 'PR-AUC lift': '{:+.4f}'})
    .set_caption(f'Baseline vs. Enriched ({best_name})')
)
print(f"\nPR-AUC lift from session_item_sim: {pr_nlp - pr_base:+.4f}")


`session_item_sim` adds marginal lift to the session classifier. The baseline behavioral features (add-to-cart rate, session duration, engagement time) already capture most of the purchasable intent within a session. 

The stronger signal for this feature appears in the Cox model (HR=3.78, p<0.0001), where focused browsing predicts how fast a user converts across sessions rather than whether a given session ends in a purchase.

## 4.4 Performance Comparison

**Why PR-AUC is the headline metric:** with ~18% positive rate, a model that pushes purchasing sessions to the top of the ranked list is commercially useful (e.g. targeting remarketing at the right users). 

PR-AUC directly measures this as it penalises a model that floods the positive class to inflate recall. ROC-AUC is shown for completeness but is less sensitive at skewed rates.

In [ ]:
# ROC and PR curves -- all models on same axes
MODEL_COLORS = {
    'Logistic Regression':  '#1565C0',
    'Random Forest':        '#2E7D32',
    'XGBoost':              '#F57F17',
    'HistGBM':              'goldenrod',
    'Ensemble (soft vote)': '#B71C1C',
}
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for name, r in results.items():
    color = MODEL_COLORS.get(name, '#888')
    style = '--' if 'Ensemble' in name else '-'
    RocCurveDisplay.from_predictions(y_test, r['probs'], ax=axes[0],
        name=f'{name} ({r["roc"]:.3f})', color=color, linestyle=style)
    PrecisionRecallDisplay.from_predictions(y_test, r['probs'], ax=axes[1],
        name=f'{name} ({r["pr"]:.3f})', color=color, linestyle=style)

axes[0].plot([0,1],[0,1],'--', color='#aaa', linewidth=0.8)
for ax, title in zip(axes, ['ROC Curves', 'Precision-Recall Curves']):
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.suptitle('Binary Session Model -- All Models', fontsize=13, fontweight='bold')
fig.tight_layout(); plt.show()

## 4.5 Feature Importance

**Permutation importance:** each feature is shuffled in turn on the held-out test set; the drop in PR-AUC is the feature's importance score. 
- Error bars come from 10 repeat shuffles. 
- This is model-agnostic and avoids the split-count bias that inflates impurity-based importance for high-cardinality categoricals (e.g. `utm_campaign`).

**SHAP (SHapley Additive exPlanations)** decomposes each individual prediction into per-feature contributions. 

Unlike permutation importance, it reveals:

- **Direction** — does a high value for this feature push toward purchase or away?
- **Magnitude** — how strongly does each feature affect that specific session's score?

The beeswarm plot shows one dot per test observation; colour = raw feature value (red = high, blue = low).

In [ ]:
# Permutation importance on best individual model (highest PR AUC, excluding ensemble)
print(f'Computing permutation importance for: {best_name}')
perm = permutation_importance(best_pipe, X_test, y_test,
                               n_repeats=10, random_state=42, scoring='average_precision')
imp_df = pd.DataFrame({
    'Feature':    feature_cols,
    'Importance': perm.importances_mean.round(4),
    'Std':        perm.importances_std.round(4)
}).sort_values('Importance', ascending=False).reset_index(drop=True)

In [ ]:
feature_names = best_pipe[:-1].get_feature_names_out()

explainer = shap.TreeExplainer(best_pipe.named_steps['clf'])
X_test_transformed = best_pipe[:-1].transform(X_test)

# Wrap in DataFrame with proper names
X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=feature_names)

shap_values = explainer.shap_values(X_test_transformed_df)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: SHAP beeswarm ---
plt.sca(ax1)
shap.summary_plot(
    shap_values, X_test_transformed_df,
    max_display=12,
    plot_size=None,          # let ax1 control the size
    show=False,
    color_bar=True,
    cmap='RdBu_r',           # cleaner diverging colormap
    alpha=0.6,               # reduce dot opacity → less muddy
)
ax1.set_title('SHAP value distribution', fontsize=12, fontweight='semibold', pad=10)
ax1.axvline(0, color='#aaa', lw=0.8, ls='--')
ax1.set_facecolor('#fafafa')
ax1.tick_params(labelsize=10)
ax1.spines[['top','right']].set_visible(False)

# --- Right: Feature importance ---
top = imp_df.head(13)
colors = ['#1d9e75' if v > 0.1 else '#5dcaa5' if v > 0.03 else '#9fe1cb' for v in top['Importance']]
ax2.barh(range(len(top)), top['Importance'][::-1].values,
         xerr=top['Std'][::-1].values,
         color=list(reversed(colors)),
         edgecolor='none', height=0.6,
         error_kw={'elinewidth': 1.2, 'ecolor': '#555', 'capsize': 3})
ax2.set_yticks(range(len(top)))
ax2.set_yticklabels(top['Feature'][::-1].values, fontsize=10)
ax2.set_xlabel('Mean drop in PR AUC when feature is shuffled', fontsize=11)
ax2.set_title('Permutation importance — HistGBM', fontsize=12, fontweight='semibold', pad=10)
ax2.axvline(0, color='#aaa', lw=0.8, ls='--')
ax2.set_facecolor('#fafafa')
ax2.spines[['top','right']].set_visible(False)
ax2.grid(True, alpha=0.25, axis='x')

plt.suptitle('Feature Analysis — Binary Session Model', fontsize=14, fontweight='semibold', y=1.01)
fig.tight_layout()
plt.show()

## 4.6 Key Findings

**Top predictors**

The top predictors are `session_duration_sec` (by a large margin), followed by `n_events`, `n_add_to_cart`, `n_view_item`, and `avg_engagement_time_msec`. 

SHAP confirms the direction: high values on all five push toward purchase.

**Notable exception**

One exception worth noting: higher `ga_session_number` is associated with lower in-session purchase probability, i.e. repeat visitors are less likely to convert in any given session, even as they are more likely to convert eventually. 

This is consistent with the Cox finding that later-session visitors take longer across the funnel.

**What doesn't matter much**
User history (`prev_purchases`, `days_since_prev_session`) and marketing variables (`utm_campaign`, `login_status`) rank near the bottom. 

Once behavioral intensity is accounted for, who the user is and how they arrived matters much less than what they do in the session.


# 5. Delayed Co-purchasing Dynamics

**Question:** If a user views a specific combination of items in session N, what is the probability those exact items are purchased together in a subsequent session N+k?

> This analysis is distinct from Market Basket Analysis (Part B): MBA finds items frequently bought together in general. 
> 
> Here we track *cross-session carry-over* — whether browsing intent formed in one session converts into a joint purchase in a later session, and how quickly that conversion happens.

## 5.1 Data Preparation

Built directly from `prod` (already filtered: no NaN or `(not set)` item IDs).

Three steps:
1. Assign a chronological **session order** to each session per user
2. Aggregate `view_item` events into per-session **viewed item sets**
3. Aggregate `purchase` events into per-session **purchased item sets**

Only sessions with ≥ 2 distinct items are kept (a pair requires at least 2 items).

In [ ]:
# ── 1. Session ordering per user ─────────────────────────────────────────────
session_starts = (
    prod.groupby(['user_pseudo_id', 'session_id'])['event_dt']
    .min().reset_index().rename(columns={'event_dt': 'session_start'})
    .sort_values(['user_pseudo_id', 'session_start'])
)
session_starts['session_order'] = (
    session_starts.groupby('user_pseudo_id').cumcount() + 1
)


In [ ]:
# ── 2. View sessions: ≥2 distinct items viewed ───────────────────────────────
view_df = prod[prod['event_name'] == 'view_item'].copy()
view_sessions_dcp = (
    view_df.groupby(['user_pseudo_id', 'session_id'])
    .agg(viewed_items    =('item_id',      lambda x: sorted(set(x))),
         viewed_cats     =('item_category', lambda x: sorted(
             {c for c in x if pd.notna(c) and c != '(not set)'})))
    .reset_index()
    .merge(session_starts[['user_pseudo_id', 'session_id', 'session_order', 'session_start']],
           on=['user_pseudo_id', 'session_id'])
)
view_sessions_dcp = view_sessions_dcp[
    view_sessions_dcp['viewed_items'].apply(len) >= 2
].copy()

In [ ]:
# ── 3. Purchase sessions: ≥2 distinct items purchased ────────────────────────
purch_df = prod[prod['event_name'] == 'purchase'].copy()
purchase_sessions_dcp = (
    purch_df.groupby(['user_pseudo_id', 'session_id'])
    .agg(purchased_items =('item_id',      lambda x: sorted(set(x))),
         purchased_cats  =('item_category', lambda x: sorted(
             {c for c in x if pd.notna(c) and c != '(not set)'})))
    .reset_index()
    .merge(session_starts[['user_pseudo_id', 'session_id', 'session_order']],
           on=['user_pseudo_id', 'session_id'])
)
purchase_sessions_dcp = purchase_sessions_dcp[
    purchase_sessions_dcp['purchased_items'].apply(len) >= 2
].copy()

print(f'View sessions with ≥2 items:      {len(view_sessions_dcp):,}')
print(f'Purchase sessions with ≥2 items:  {len(purchase_sessions_dcp):,}')
print(f'Users with multi-item purchases:  {purchase_sessions_dcp["user_pseudo_id"].nunique():,}')

## 5.2 Item-level Delayed Co-purchase

For every session with ≥ 2 viewed items, enumerate all item pairs and check whether that exact pair appears together in a later purchase session by the same user. `lag_sessions` records how many sessions later the co-purchase happened.

In [ ]:
# ── Enumerate view pairs ─────────────────────────────────────────────────────
view_pair_rows = []
for _, row in view_sessions_dcp.iterrows():
    for a, b in combinations(row['viewed_items'], 2):
        if a > b: a, b = b, a
        view_pair_rows.append({
            'user_pseudo_id':  row['user_pseudo_id'],
            'view_session_id': row['session_id'],
            'view_order':      row['session_order'],
            'item_a': a, 'item_b': b,
            'pair_key': f'{a} | {b}'
        })
view_pairs_dcp = pd.DataFrame(view_pair_rows)

# ── Purchase lookup: user → [(session_order, frozenset of items)] ─────────────
purch_item_lookup = {}
for _, row in purchase_sessions_dcp.iterrows():
    purch_item_lookup.setdefault(row['user_pseudo_id'], []).append(
        (row['session_order'], frozenset(row['purchased_items']))
    )

# ── Cross-session matching ────────────────────────────────────────────────────
later_cp, lag_cp = [], []
for _, row in view_pairs_dcp.iterrows():
    pair = frozenset([row['item_a'], row['item_b']])
    found, lag = 0, np.nan
    for pord, pitems in purch_item_lookup.get(row['user_pseudo_id'], []):
        if pord > row['view_order'] and pair.issubset(pitems):
            found, lag = 1, pord - row['view_order']
            break
    later_cp.append(found)
    lag_cp.append(lag)

view_pairs_dcp['later_copurchased'] = later_cp
view_pairs_dcp['lag_sessions']      = lag_cp

overall_item_prob = view_pairs_dcp['later_copurchased'].mean()
print(f'Total view pairs:                 {len(view_pairs_dcp):,}')
print(f'Pairs later co-purchased:         {int(view_pairs_dcp["later_copurchased"].sum()):,}')
print(f'Overall delayed co-purchase rate: {overall_item_prob:.4f} ({overall_item_prob*100:.2f}%)')


In [ ]:
# ── Item-pair summary table (≥5 observations, show only pairs with ≥1 co-purchase) ──
item_summary = (
    view_pairs_dcp.groupby('pair_key', as_index=False)
    .agg(n_viewed =('pair_key',        'size'),
         n_bought  =('later_copurchased','sum'),
         avg_lag   =('lag_sessions',    'mean'))
)
item_summary['prob'] = item_summary['n_bought'] / item_summary['n_viewed']

top_items = (
    item_summary[(item_summary['n_viewed'] >= 5) & (item_summary['n_bought'] > 0)]
    .sort_values(['n_bought', 'prob'], ascending=False)
    .head(15)
    .rename(columns={'pair_key': 'Item Pair', 'n_viewed': 'Viewed Together',
                     'n_bought': 'Later Co-purchased', 'avg_lag': 'Avg Lag (sessions)',
                     'prob': 'Co-purchase Prob'})
)
display(top_items.style.format({
    'Co-purchase Prob':    '{:.1%}',
    'Avg Lag (sessions)': '{:.1f}'
}).set_caption('Top item pairs: delayed co-purchase (≥5 co-views, ≥1 co-purchase)'))


**Item-level takeaway:** 
- Delayed co-purchase at the exact item level is rare (well under 1% of viewed pairs). 
- The few high-probability pairs tend to be near-identical SKUs (e.g. same style in different colours or sizes), where the user viewed both variants and returned to buy them together. 

    This sparsity motivates the category-level analysis below.

## 5.3 Category-level Delayed Co-purchase

Collapsing to product category reduces sparsity and reveals broader intent patterns. 

A user who viewed *women's blazers* and *women's heels* together, and later bought both categories, is captured here even if they didn't return for the exact same SKUs.

In [ ]:
# ── Enumerate category pairs from view sessions ───────────────────────────────
view_cat_rows = []
for _, row in view_sessions_dcp.iterrows():
    cats = row['viewed_cats']
    if len(cats) < 2:
        continue
    for a, b in combinations(cats, 2):
        if a > b: a, b = b, a
        view_cat_rows.append({
            'user_pseudo_id': row['user_pseudo_id'],
            'view_order':     row['session_order'],
            'cat_a': a, 'cat_b': b,
            'cat_pair_key': f'{a} | {b}'
        })
view_cat_pairs = pd.DataFrame(view_cat_rows)

# ── Purchase lookup at category level ─────────────────────────────────────────
purch_cat_lookup = {}
for _, row in purchase_sessions_dcp.iterrows():
    purch_cat_lookup.setdefault(row['user_pseudo_id'], []).append(
        (row['session_order'], frozenset(row['purchased_cats']))
    )

# ── Cross-session matching (category level) ───────────────────────────────────
later_cat_cp, lag_cat_cp = [], []
for _, row in view_cat_pairs.iterrows():
    pair = frozenset([row['cat_a'], row['cat_b']])
    found, lag = 0, np.nan
    for pord, pcats in purch_cat_lookup.get(row['user_pseudo_id'], []):
        if pord > row['view_order'] and pair.issubset(pcats):
            found, lag = 1, pord - row['view_order']
            break
    later_cat_cp.append(found)
    lag_cat_cp.append(lag)

view_cat_pairs['later_copurchased'] = later_cat_cp
view_cat_pairs['lag_sessions']      = lag_cat_cp

overall_cat_prob = view_cat_pairs['later_copurchased'].mean()
print(f'Total category pairs:             {len(view_cat_pairs):,}')
print(f'Pairs later co-purchased:         {int(view_cat_pairs["later_copurchased"].sum()):,}')
print(f'Overall delayed co-purchase rate: {overall_cat_prob:.4f} ({overall_cat_prob*100:.2f}%)')

# ── Category-pair summary (≥5 observations, at least 1 co-purchase) ──────────
cat_summary = (
    view_cat_pairs.groupby('cat_pair_key', as_index=False)
    .agg(n_viewed =('cat_pair_key',    'size'),
         n_bought  =('later_copurchased','sum'),
         avg_lag   =('lag_sessions',    'mean'))
)
cat_summary['prob'] = cat_summary['n_bought'] / cat_summary['n_viewed']

top_cats = (
    cat_summary[(cat_summary['n_viewed'] >= 5) & (cat_summary['n_bought'] > 0)]
    .sort_values(['n_bought', 'prob'], ascending=False)
    .head(15)
    .rename(columns={'cat_pair_key': 'Category Pair', 'n_viewed': 'Viewed Together',
                     'n_bought': 'Later Co-purchased', 'avg_lag': 'Avg Lag (sessions)',
                     'prob': 'Co-purchase Prob'})
)
display(top_cats.style.format({
    'Co-purchase Prob':    '{:.1%}',
    'Avg Lag (sessions)': '{:.1f}'
}).set_caption('Top category pairs: delayed co-purchase (≥5 co-views, ≥1 co-purchase)'))


**Category-level takeaway:** 
- Co-purchase rates are meaningfully higher at the category level than the item level — users maintain *category-level preferences* across sessions even when they do not return for the exact same product. 
- These category combinations are the most actionable signal for cross-selling and remarketing targeting.

## 5.4 Lag-k Breakdown

> How quickly does the delayed co-purchase happen? 

We compute the cumulative co-purchase probability as a function of session lag k and plot the distribution of first co-purchase lags.

In [ ]:
# ── Cumulative co-purchase probability by lag k (item level) ─────────────────
max_k = int(view_pairs_dcp['lag_sessions'].dropna().max()) if view_pairs_dcp['later_copurchased'].sum() > 0 else 10
max_k = min(max_k, 15)

lag_k_rows = []
for k in range(1, max_k + 1):
    n_within_k = (view_pairs_dcp['lag_sessions'] <= k).sum()
    lag_k_rows.append({
        'Lag k': k,
        'New co-purchases at lag k': (view_pairs_dcp['lag_sessions'] == k).sum(),
        'Cumulative co-purchases':   n_within_k,
        'Cumulative rate (%)':       round(n_within_k / len(view_pairs_dcp) * 100, 4)
    })
lag_k_df = pd.DataFrame(lag_k_rows)
display(lag_k_df)

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of lag at first co-purchase
lag_vals = view_pairs_dcp['lag_sessions'].dropna().astype(int)
if len(lag_vals) > 0:
    counts = lag_vals.value_counts().sort_index()
    ax1.bar(counts.index, counts.values, color='#1A237E', edgecolor='white', width=0.7)
    ax1.set_xlabel('Sessions until co-purchase (lag k)', fontsize=11)
    ax1.set_ylabel('Number of pairs', fontsize=11)
    ax1.set_title('Distribution of first co-purchase lag', fontsize=12, fontweight='semibold')
    ax1.set_xticks(counts.index)
else:
    ax1.text(0.5, 0.5, 'No co-purchases observed', ha='center', va='center', transform=ax1.transAxes)

# Right: cumulative probability curve
ax2.plot(lag_k_df['Lag k'], lag_k_df['Cumulative rate (%)'], 'o-', color='#1A237E', linewidth=2)
ax2.fill_between(lag_k_df['Lag k'], lag_k_df['Cumulative rate (%)'], alpha=0.15, color='#1A237E')
ax2.set_xlabel('Maximum session lag k', fontsize=11)
ax2.set_ylabel('Cumulative co-purchase rate (%)', fontsize=11)
ax2.set_title('Cumulative co-purchase probability by lag', fontsize=12, fontweight='semibold')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.3f}%'))

plt.tight_layout()
plt.show()


**Lag takeaway:** 
- Co-purchases that do occur are concentrated at lag 1 (73 pairs) with a secondary cluster around lag 4 (46 pairs), then almost nothing beyond lag 6. 
- The two-wave pattern is suggestive of two distinct behaviours: immediate return visits and slightly delayed consideration 

    However, with 180 total events the lag distribution is too sparse to draw firm conclusions about timing.

## 5.5 LLM Pair Relationship Feature

We use OpenAI `gpt-4o-mini` to classify every unique co-viewed item pair into one of four categories:

| Label | Meaning | Example |
|---|---|---|
| **COMPLEMENTARY** | Different product types worn together | Bag + shoes, jacket + shirt |
| **SUBSTITUTE** | Same product type, alternatives | Two similar handbags |
| **STYLE_MATCH** | Different types, same aesthetic/collection/color | Baroque bag + baroque scarf |
| **UNRELATED** | No clear fashion relationship | Sneakers + home decor |

Item descriptions are built from `operative_brand`, `commercial_category`, `commercial_sub_category`,
`style_fabric_color`, and `item_category` — giving the LLM enough context to detect style alignment.

Results are saved to `pair_relationships.csv` (checkpoint) so classification runs only once.
The label is used as a feature in the delayed co-purchase predictive model (§4.7).


In [ ]:
# import os, time
# from dotenv import load_dotenv
# from openai import OpenAI

# load_dotenv()
# client_oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# # ── Build item description map ────────────────────────────────────────────────
# _mba = pd.read_csv("IMA2026_Versace_MBA_Product_Level.csv", low_memory=False)
# _mba_meta = (_mba[["item_id","operative_brand","commercial_category",
#                     "commercial_sub_category","style_fabric_color"]]
#              .drop_duplicates("item_id"))
# _ttp_meta = (prod[prod["item_id"].notna() & (prod["item_id"] != "(not set)")]
#              [["item_id","item_category"]].drop_duplicates("item_id"))
# _item_meta = _ttp_meta.merge(_mba_meta, on="item_id", how="left")

# def _build_desc(item_id):
#     rows = _item_meta[_item_meta["item_id"] == item_id]
#     if rows.empty: return item_id
#     r = rows.iloc[0]
#     parts = [str(r[c]).strip() for c in
#              ["operative_brand","commercial_category","commercial_sub_category","style_fabric_color"]
#              if pd.notna(r.get(c)) and str(r.get(c)).strip()]
#     cat = r.get("item_category")
#     if pd.notna(cat) and str(cat) not in ("(not set)", ""):
#         parts.append(str(cat).strip())
#     return " | ".join(parts) if parts else item_id

# # ── Batch LLM classifier (20 pairs per API call) ──────────────────────────────
# LABELS = {"COMPLEMENTARY", "SUBSTITUTE", "STYLE_MATCH", "UNRELATED"}
# BATCH_SIZE = 20

# def classify_batch(pairs):
#     lines = [f"{i}. A: {_build_desc(a)} || B: {_build_desc(b)}"
#              for i, (a, b) in enumerate(pairs, 1)]
#     prompt = (
#         "You are a luxury fashion expert for Versace.\n"
#         "Classify each product pair as one of: COMPLEMENTARY, SUBSTITUTE, STYLE_MATCH, UNRELATED.\n"
#         "- COMPLEMENTARY: different types worn together (bag + shoes, jacket + shirt)\n"
#         "- SUBSTITUTE: same type, same need (two similar handbags)\n"
#         "- STYLE_MATCH: different types, same aesthetic/collection/color story\n"
#         "- UNRELATED: no clear fashion relationship\n\n"
#         "Reply with ONLY a comma-separated list of labels in the same order, nothing else.\n\n"
#         + "\n".join(lines)
#     )
#     for attempt in range(3):
#         try:
#             resp = client_oai.chat.completions.create(
#                 model="gpt-4o-mini",
#                 messages=[{"role": "user", "content": prompt}],
#                 temperature=0, max_tokens=len(pairs) * 8
#             )
#             raw    = resp.choices[0].message.content.strip().upper()
#             labels = [l.strip() for l in raw.split(",")]
#             labels = [l if l in LABELS else "UNRELATED" for l in labels]
#             if len(labels) < len(pairs):
#                 labels += ["UNRELATED"] * (len(pairs) - len(labels))
#             return labels[:len(pairs)]
#         except Exception as e:
#             wait = 30 if "429" in str(e) else 5
#             print(f"  Error attempt {attempt+1}: {str(e)[:60]}, waiting {wait}s")
#             time.sleep(wait)
#     return ["UNRELATED"] * len(pairs)

# # ── Load checkpoint ───────────────────────────────────────────────────────────
# CHECKPOINT = "pair_relationships.csv"
# unique_pairs = view_pairs_dcp[["item_a","item_b","pair_key"]].drop_duplicates("pair_key")

# if os.path.exists(CHECKPOINT):
#     classified = pd.read_csv(CHECKPOINT)
#     done_keys  = set(classified["pair_key"])
#     print(f"Checkpoint loaded: {len(classified):,} pairs already classified")
# else:
#     classified = pd.DataFrame(columns=["pair_key","relationship"])
#     done_keys  = set()

# remaining = unique_pairs[~unique_pairs["pair_key"].isin(done_keys)].copy().reset_index(drop=True)
# n_batches  = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE
# print(f"Pairs to classify: {len(remaining):,} | Batches: {n_batches:,} (x{BATCH_SIZE}/batch)")
# print(f"Estimated time:    ~{n_batches * 2 / 60:.0f} min | cost ~${len(remaining)*200/1e6*0.15:.2f}")

# # ── Classify ─────────────────────────────────────────────────────────────────
# new_rows = []
# for batch_i in range(n_batches):
#     batch_df = remaining.iloc[batch_i*BATCH_SIZE : (batch_i+1)*BATCH_SIZE]
#     pairs_in = list(zip(batch_df["item_a"], batch_df["item_b"]))
#     labels   = classify_batch(pairs_in)
#     for (_, row), label in zip(batch_df.iterrows(), labels):
#         new_rows.append({"pair_key": row["pair_key"], "relationship": label})
#     time.sleep(0.15)

#     if (batch_i + 1) % 25 == 0:  # checkpoint every 500 pairs
#         classified = pd.concat([classified, pd.DataFrame(new_rows)], ignore_index=True)
#         classified.to_csv(CHECKPOINT, index=False)
#         new_rows = []
#         print(f"  {(batch_i+1)*BATCH_SIZE:,}/{len(remaining):,} pairs done — checkpoint saved")

# if new_rows:
#     classified = pd.concat([classified, pd.DataFrame(new_rows)], ignore_index=True)
#     classified.to_csv(CHECKPOINT, index=False)

# print(f"Done. {len(classified):,} unique pairs classified.")

# # ── Join back and show distribution ──────────────────────────────────────────
# view_pairs_dcp = view_pairs_dcp.merge(classified, on="pair_key", how="left")
# view_pairs_dcp["relationship"] = view_pairs_dcp["relationship"].fillna("UNRELATED")

# print("\nRelationship distribution:")
# display(view_pairs_dcp["relationship"].value_counts()
#         .rename_axis("Label").reset_index(name="Count"))

# print("\nCo-purchase rate by relationship type:")
# display(
#     view_pairs_dcp.groupby("relationship")["later_copurchased"]
#     .agg(n_pairs="count", copurchases="sum", rate="mean")
#     .sort_values("rate", ascending=False)
#     .style.format({"rate": "{:.4%}"})
# )


In [ ]:
# ── Load pair relationships from CSV (run this instead of the LLM cell above) ──
CHECKPOINT = "pair_relationships.csv"

if "relationship" not in view_pairs_dcp.columns:
    if os.path.exists(CHECKPOINT):
        _rel = pd.read_csv(CHECKPOINT)
        view_pairs_dcp = view_pairs_dcp.merge(_rel, on="pair_key", how="left")
        view_pairs_dcp["relationship"] = view_pairs_dcp["relationship"].fillna("UNRELATED")
        print(f"Loaded {len(_rel):,} pair relationships from {CHECKPOINT}")
    else:
        print("No CSV found — run the LLM cell above first.")
        view_pairs_dcp["relationship"] = "UNRELATED"
else:
    print("relationship column already in view_pairs_dcp — skipping load")

print(view_pairs_dcp["relationship"].value_counts().to_string())


## 5.6 Predictive Model: Will This Viewed Pair Be Co-purchased?

**Target:** `later_copurchased` (1 = pair bought together in a later session)

**Features:**
- `view_order` — how early in the session the pair appeared
- `session_item_sim` — browsing focus signal (NLP cosine similarity, from §2.3)
- `n_items_session` — number of items viewed in the session
- `relationship` dummies — LLM fashion logic (COMPLEMENTARY / SUBSTITUTE / STYLE_MATCH vs UNRELATED baseline)

**NOTE:** 
- ROC-AUC with GroupKFold (5 splits) on `user_pseudo_id` to prevent cross-user leakage.
- Positive rate is ~1% (180 co-purchased pairs out of ~18k view pairs). 
- ROC-AUC is reported alongside caveats on statistical power. - Baseline uses behavioral features only; enriched adds the LLM relationship label.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb

# ── Feature engineering ───────────────────────────────────────────────────────
model_pairs = view_pairs_dcp.copy()

if "session_item_sim" in session_viewed.columns:
    model_pairs = model_pairs.merge(
        session_viewed[["session_id","session_item_sim"]]
        .rename(columns={"session_id":"view_session_id"}),
        on="view_session_id", how="left"
    )
    _med = model_pairs["session_item_sim"].median()
    model_pairs["session_item_sim"] = model_pairs["session_item_sim"].fillna(
        _med if pd.notna(_med) else 0)
else:
    print("Warning: session_item_sim not available — run §2.3 first. Using 0.")
    model_pairs["session_item_sim"] = 0.0

_sess_size = view_sessions_dcp[["session_id","viewed_items"]].copy()
_sess_size["n_items_session"] = _sess_size["viewed_items"].apply(len)
model_pairs = model_pairs.merge(
    _sess_size[["session_id","n_items_session"]].rename(columns={"session_id":"view_session_id"}),
    on="view_session_id", how="left"
)
model_pairs["n_items_session"] = model_pairs["n_items_session"].fillna(2)

if "relationship" in model_pairs.columns:
    rel_dummies = pd.get_dummies(model_pairs["relationship"], prefix="rel")
    if "rel_UNRELATED" in rel_dummies.columns:
        rel_dummies = rel_dummies.drop(columns=["rel_UNRELATED"])
    rel_cols = rel_dummies.columns.tolist()
    model_pairs = pd.concat([model_pairs, rel_dummies], axis=1)
else:
    print("Warning: relationship column missing — run load cell above first.")
    rel_cols = []

base_features = ["view_order", "session_item_sim", "n_items_session"]
enr_features  = base_features + rel_cols
y_dcp         = model_pairs["later_copurchased"].astype(int)
groups_dcp    = model_pairs["user_pseudo_id"]

# ── Diagnostics ──────────────────────────────────────────────────────────────
print(f"Rows: {len(model_pairs):,} | Positives: {y_dcp.sum()} ({y_dcp.mean():.2%})")
print(f"NaN in features:\n{model_pairs[enr_features].isna().sum().to_string()}")

# ── Models ────────────────────────────────────────────────────────────────────
models = {
    "LogReg": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000))
    ]),
    "LightGBM": lgb.LGBMClassifier(
        is_unbalance=True, n_estimators=300, random_state=42,
        verbosity=-1, n_jobs=-1),
}

gkf    = StratifiedGroupKFold(n_splits=5)
scorer = "roc_auc"

rows = []
for name, model in models.items():
    s_base = cross_val_score(model, model_pairs[base_features].fillna(0),
                             y_dcp, groups=groups_dcp, cv=gkf,
                             scoring=scorer, error_score="raise")
    s_enr  = cross_val_score(model, model_pairs[enr_features].fillna(0),
                             y_dcp, groups=groups_dcp, cv=gkf,
                             scoring=scorer, error_score="raise")
    rows.append({"Model": name,
                 "Baseline ROC-AUC": round(s_base.mean(), 4),
                 "Enriched ROC-AUC": round(s_enr.mean(), 4),
                 "Lift":             round(s_enr.mean() - s_base.mean(), 4)})
    print(f"{name:15s}  base={s_base.mean():.4f}  enr={s_enr.mean():.4f}  "
          f"lift={s_enr.mean()-s_base.mean():+.4f}")

comp_dcp = pd.DataFrame(rows).sort_values("Enriched ROC-AUC", ascending=False)
display(comp_dcp.style.format(
    {"Baseline ROC-AUC": "{:.4f}", "Enriched ROC-AUC": "{:.4f}", "Lift": "{:+.4f}"}))

# ── Feature importances from best model ───────────────────────────────────────
best_name_dcp = comp_dcp.iloc[0]["Model"]
best_model    = models[best_name_dcp]
best_model.fit(model_pairs[enr_features].fillna(0), y_dcp)

if hasattr(best_model, "named_steps"):
    importances = best_model.named_steps["clf"].coef_[0]
elif hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
else:
    importances = None

if importances is not None:
    imp_df = pd.DataFrame({"Feature": enr_features, "Importance": importances})\
               .sort_values("Importance", ascending=False)
    display(imp_df.style
        .format({"Importance": "{:+.4f}"})
        .bar(subset=["Importance"], align="zero", color=["#E53935","#43A047"])
        .set_caption(f"Feature importances — {best_name_dcp}"))


## 5.7 Key Findings

**Overall rates**

- Delayed co-purchase rate: under 1% at item level, 2.2% at category level
- Low in absolute terms, but consistent: browsing intent formed in one session does carry over into purchase in a later session for a meaningful minority of users

**Category pairs are more actionable than item pairs**

- Purchase intent is category-stable: users return to buy within the same category even when they switch SKU
- Top pairs - blazers + skirts, clothing + shoes, children's clothing + gift sets, bathrobes + slippers are coherent outfit or lifestyle bundles and represent the clearest cross-sell opportunities

**What the predictive model adds**

- Session focus (session_item_sim) is the strongest positive predictor — users browsing similar items together are more likely to return and buy them as a pair
- Relationship type matters: complementary pairs (different product types worn together) are more likely to be co-purchased than substitutes (two versions of the same product) — substitutes tend to resolve into a single purchase, not a joint one

> Caveat on statistical power
> 
> - All results rest on 180 co-purchased pairs out of 18,115 viewed pairs
> - Category-level and model findings are directionally credible but not stable enough to act on at the individual item level
> - Treat these as signals to inform cross-sell design and retargeting logic, not precise probability estimates

# 6. Findings & Recommendations



## 6.1. Acquisition
**Channel drives discovery, not conversion**

- Marketing variables (UTM source, campaign type) rank near the bottom of feature importance in both the survival and session models - once a user is on-site, how they arrived matters far less than what they do Sale campaigns accelerate conversion for low-price, mobile, first-time visitors (the Spontaneous archetype) - but have no meaningful effect on mid-to-high price segments 
- Retargeting is most effective when it re-engages users who already have a purchase history - for first-time visitors, it does not replicate the same conversion speed

> **Recommendation**
> 
> Do not judge campaign effectiveness by last-click conversion alone — the value of paid channels is in surfacing the right user, not closing the sale
> 
> Retargeting budget is better spent on recent buyers (Humanistic archetype) than on cold first-time visitors

## 6.2. On-Site Experience
**Browsing focus is the strongest behavioral signal**

- Users who browse a narrow, coherent set of items convert faster and are more likely to return for a multi-item purchase
- Scattered browsing across unrelated categories is a reliable non-conversion signal — these users are in discovery mode, not shopping mode
- Session duration, number of events, and engagement time are the top predictors of in-session conversion, by a large margin over any user characteristic

**The conversion window is narrow**

- Most purchases happen within 3 days of first item view - after that the probability of conversion drops sharply and does not recover
- Users who don't convert quickly are unlikely to convert at all within the same consideration cycle

> **Recommendation**
> 
> - When a session shows focused browsing (similar items, coherent category), reduce friction immediately: surface checkout prominently, offer styling bundles, minimize navigation away from the funnel
> - When browsing is scattered, prioritize editorial content and category guidance over conversion pressure: pushing checkout too early will not work
> 
> - Invest in product page depth for the Methodical archetype (desktop, many sessions, mid-range price) - these users return repeatedly and need informational richness to move forward

## 6.3. Conversion
**Behavioral intensity separates buyers from browsers**

- Add-to-cart is the clearest in-session commitment signal - sessions with add-to-cart events are disproportionately likely to convert
- Repeat visitors (higher session number) are less likely to convert in any given session, but more likely to convert eventually - they are not lost, they are deliberating

> **Recommendation**
> 
> - Trigger recovery sequences within day 1–2 post first-view, not the standard 7–30 day windows — the window closes fast
> - Prioritize live support (chat, styling advice) for high-intensity sessions that have added to cart but not checked out — these are the users closest to converting
> - Tailor the experience to the archetype:
>   - Spontaneous → streamlined mobile checkout, urgency cue, no distraction
>   - Humanistic → personalized greeting, recently viewed items, loyalty acknowledgment
>   - Methodical → rich product detail, editorial context, comparison tools
>   - Competitive → craftsmanship storytelling, exclusivity framing

## 6.4. Retention & Cross-Sell
**Cross-session intent carry-over exists at the category level**

- Delayed co-purchase rate is under 1% at the item level but rises to 2.2% at the category level: users return to buy within the same category even when they switch SKU
- The most actionable category pairs are: blazers + skirts, tops + sneakers, children's clothing + gift sets, bathrobes + slippers
- Complementary pairs (different product types worn together) are more likely to result in a joint purchase than substitutes (two versions of the same product): substitutes tend to resolve into a single purchase

**Prior purchase history accelerates conversion**

Recent buyers convert faster across all models — the brand relationship is already established

Note: given the ~2 month observation window, this reflects recency rather than long-term loyalty

> **Recommendation**
> 
> - Build "complete the look" modules and post-purchase email sequences around category pairs, not individual SKUs — the signal is more stable and actionable at that level
> - Focus cross-sell on complementary pairs only — do not push substitute items as cross-sell targets
> - For recent buyers, prioritize re-engagement within a short window — the conversion speed advantage decays with time